# GPU test 03 — scene (room) capture

**Goal:** train a whole-room splat from a *Scene capture* made by our Android app (4–6 spots, phone at
arm's length). Uses the `scene` training profile: 30k steps, images at 1600 px, up to 3M Gaussians,
plus monocular depth priors, MCMC densification, per-photo exposure and pose refinement against holes and
"fur" on blank walls and ceilings.

**Before running:** upload the capture `.zip` (ideally the **refined** one we made on the PC, so this
notebook skips the slow CPU pose refinement) as a **private Kaggle Dataset**, then *Add Input* → it.

**Kaggle settings:** Accelerator = **GPU T4 x2** · Internet = **On**. Then **Run All**
(~70–100 min; add ~20 min if the capture still needs pose refinement).

**Report back:** the metrics JSON (especially `holes.hole_fraction`), the preview images, and download
`out/splat_object.ply` (the room: floor at y = 0, cropped to the room) — open it in our viewer.
`previews/holes_station<k>.png` show the view from each capture spot as a cubemap
(front right back left / up down); magenta = see-through holes = places to re-shoot.

> Generated by `worker/build_gpu_notebooks.py` — don't edit by hand.

## 1. Environment check

In [ ]:
import sys, shutil, subprocess, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2"
name, cap = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
print("GPU:", name, "compute capability", cap)
if cap < (7, 0):
    print("WARNING: this GPU is too old for current PyTorch/gsplat. Switch Accelerator to 'GPU T4 x2'.")
nvcc = shutil.which("nvcc") or ("/usr/local/cuda/bin/nvcc" if shutil.os.path.exists("/usr/local/cuda/bin/nvcc") else None)
print("nvcc:", nvcc)
if nvcc:
    print(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1])

## 2. Install gsplat + compile CUDA kernels

In [ ]:
# gsplat ships as source and compiles its CUDA kernels on first use (~3-8 min, once per session).
!pip install -q --no-deps gsplat==1.5.3
!pip install -q jaxtyping rich ninja remotezip jsonschema

In [ ]:
import os, time, torch
maj, mnr = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{maj}.{mnr}"          # compile only for this GPU (faster)
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
t = time.time()
import gsplat
from gsplat import rasterization
N = 2000
means = torch.randn(N, 3, device="cuda") * 0.5 + torch.tensor([0, 0, 4.0], device="cuda")
quats = torch.nn.functional.normalize(torch.randn(N, 4, device="cuda"), dim=-1)
scales = torch.full((N, 3), 0.03, device="cuda")
opac = torch.full((N,), 0.8, device="cuda")
colors = torch.rand(N, 3, device="cuda")
K = torch.tensor([[100.0, 0, 64], [0, 100.0, 64], [0, 0, 1]], device="cuda")
img, alpha, meta = rasterization(means, quats, scales, opac, colors, torch.eye(4, device="cuda")[None], K[None], 128, 128)
torch.cuda.synchronize()
print(f"gsplat {gsplat.__version__} CUDA OK: rendered {tuple(img.shape)}, mean alpha {alpha.mean():.3f}, "
      f"compile+render {time.time() - t:.0f}s")

## 3. Write our pipeline code (`gs_pipeline`, generated from the repo)

In [ ]:
import os
os.makedirs('/kaggle/working/gs_pipeline', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/gs_pipeline/__init__.py
"""Shared capture-bundle + Gaussian splat training pipeline.

Pure-numpy modules (cameras, manifest, colmap_io, ply_io, validate, visualize)
run anywhere. Only `train` needs torch + a CUDA GPU.
"""

__version__ = "0.1.0"
SCHEMA_VERSION = "1.0"

In [ ]:
%%writefile /kaggle/working/gs_pipeline/__main__.py
"""CLI: python -m gs_pipeline {convert-colmap,validate,plot,train} ..."""

import argparse
import sys


def main(argv=None) -> int:
    p = argparse.ArgumentParser(prog="gs_pipeline")
    sub = p.add_subparsers(dest="cmd", required=True)

    c = sub.add_parser("convert-colmap", help="COLMAP model -> manifest.json")
    c.add_argument("scene_dir")
    c.add_argument("--images", default="images", help="image subfolder, e.g. images_4")
    c.add_argument("--sparse", default="sparse/0")

    v = sub.add_parser("validate", help="CPU sanity checks on a capture bundle")
    v.add_argument("manifest")
    v.add_argument("--no-files", action="store_true", help="skip image existence/size checks")

    pl = sub.add_parser("plot", help="camera frustum plot -> PNG")
    pl.add_argument("manifest")
    pl.add_argument("--out", default="cameras.png")

    rf = sub.add_parser("refine", help="COLMAP pose/lens refinement aligned to the AR world (CPU)")
    rf.add_argument("manifest")
    rf.add_argument("--out", required=True, help="output folder for the refined bundle")
    rf.add_argument("--threads", type=int, default=2)

    pc = sub.add_parser("posecheck", help="epipolar consistency of poses vs photos, in px (CPU, OpenCV)")
    pc.add_argument("manifest")

    ov = sub.add_parser("overlay", help="draw AR points + object bounds onto photos (pose/orientation check)")
    ov.add_argument("manifest")
    ov.add_argument("--out", default="overlay.png")
    ov.add_argument("--frames", type=int, default=6)

    pp = sub.add_parser("postprocess", help="upright + crop a trained splat to the object (CPU)")
    pp.add_argument("splat_ply")
    pp.add_argument("manifest")
    pp.add_argument("--out", default="splat_object.ply")
    pp.add_argument("--radius", type=float, default=None, help="crop radius in scene units (default: auto)")
    pp.add_argument("--min-opacity", type=float, default=0.02)

    dp = sub.add_parser("depth", help="monocular depth priors for room captures (GPU recommended)")
    dp.add_argument("manifest")
    dp.add_argument("--out", default=None, help="output folder (default: <bundle>/depth)")
    dp.add_argument("--model", default=None, help="HF depth model (default: Depth-Anything-V2-Large)")

    hl = sub.add_parser("holes", help="hole report of an existing splat.ply from the capture spots (needs CUDA)")
    hl.add_argument("splat_ply", help="the trainer's splat.ply (not splat_object.ply)")
    hl.add_argument("manifest")
    hl.add_argument("--out", default="holes")

    t = sub.add_parser("train", help="train a Gaussian splat (needs CUDA)")
    from .train_config import add_train_args

    add_train_args(t)

    args = p.parse_args(argv)
    if args.cmd == "convert-colmap":
        from .colmap_io import convert_colmap

        print(convert_colmap(args.scene_dir, args.images, args.sparse))
    elif args.cmd == "validate":
        from .validate import validate_manifest

        rep = validate_manifest(args.manifest, check_files=not args.no_files)
        print(rep.summary())
        return 0 if rep.ok else 1
    elif args.cmd == "plot":
        from .visualize import plot_cameras

        print(plot_cameras(args.manifest, args.out))
    elif args.cmd == "refine":
        import json

        from .refine import refine

        report = refine(args.manifest, args.out, num_threads=args.threads)
        print(json.dumps(report, indent=2))
    elif args.cmd == "posecheck":
        import json

        from .posecheck import posecheck

        print(json.dumps(posecheck(args.manifest), indent=2))
    elif args.cmd == "overlay":
        from .overlay import overlay

        print(overlay(args.manifest, args.out, args.frames))
    elif args.cmd == "postprocess":
        from .postprocess import postprocess

        print(postprocess(args.splat_ply, args.manifest, args.out, args.radius, args.min_opacity))
    elif args.cmd == "depth":
        import json

        from .depth import MODEL, compute_depth

        print(json.dumps(compute_depth(args.manifest, args.out, args.model or MODEL), indent=2))
    elif args.cmd == "holes":
        import json

        from .train import holes_for_splat

        print(json.dumps(holes_for_splat(args.splat_ply, args.manifest, args.out), indent=2))
    elif args.cmd == "train":
        from .train import train

        train(args)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /kaggle/working/gs_pipeline/bounds.py
"""Where the object actually is.

The app's object_bounds is a cylinder the user tapped onto an early AR plane estimate, so it can be
off by 10+ cm (seen on the first OnePlus capture). The AR feature points are measured, so we refine
the bounds from the points that stand up off the floor near the tapped spot.
"""

from dataclasses import dataclass

import numpy as np

MIN_OBJECT_POINTS = 50


@dataclass
class ObjectBounds:
    base: np.ndarray  # footprint center on the floor (y up)
    radius: float
    height: float
    source: str       # "refined from N points" | "user cylinder"

    @property
    def middle(self) -> np.ndarray:
        return self.base + [0.0, 0.5 * self.height, 0.0]

    @property
    def sphere_radius(self) -> float:
        return float(np.hypot(self.radius, 0.5 * self.height))


def user_bounds(manifest: dict) -> ObjectBounds | None:
    b = manifest.get("object_bounds")
    if not b:
        return None
    return ObjectBounds(np.asarray(b["center"], float), float(b["radius"]),
                        float(b.get("height", b["radius"])), "user cylinder")


def refine_bounds(manifest: dict, xyz: np.ndarray | None) -> ObjectBounds | None:
    """Refine the user cylinder with AR points; falls back to the user cylinder if points are too sparse."""
    user = user_bounds(manifest)
    if user is None or xyz is None or len(xyz) == 0:
        return user
    horiz = np.hypot(xyz[:, 0] - user.base[0], xyz[:, 2] - user.base[2])

    # Floor = densest 1 cm height layer over a wide area (near the object, the object's own points can
    # outnumber the floor's). The AR plane the user tapped can be 10+ cm off, so search +-0.6 m around it.
    wide = xyz[(horiz < 1.5) & (np.abs(xyz[:, 1] - user.base[1]) < 0.6)]
    if len(wide) < MIN_OBJECT_POINTS:
        return user
    edges = np.arange(wide[:, 1].min(), wide[:, 1].max() + 0.02, 0.01)
    hist, edges = np.histogram(wide[:, 1], bins=edges)
    floor = float(edges[int(np.argmax(np.convolve(hist, np.ones(3) / 3, mode="same")))] + 0.005)

    # The object = points clearly above the floor near the tapped spot.
    search = max(2.5 * user.radius, user.radius + 0.15)
    top = floor + max(2 * user.height, 0.6)
    obj = xyz[(horiz < search) & (xyz[:, 1] > floor + 0.03) & (xyz[:, 1] < top)]
    if len(obj) < MIN_OBJECT_POINTS:
        return user

    center_xz = np.median(obj[:, [0, 2]], axis=0)
    r = float(np.percentile(np.hypot(obj[:, 0] - center_xz[0], obj[:, 2] - center_xz[1]), 90))
    h = float(np.percentile(obj[:, 1], 95) - floor)
    return ObjectBounds(np.array([center_xz[0], floor, center_xz[1]]), max(r, 0.5 * user.radius),
                        max(h, 0.5 * user.height), f"refined from {len(obj)} points")

In [ ]:
%%writefile /kaggle/working/gs_pipeline/cameras.py
"""Camera conventions — the single place where coordinate frames are converted.

Manifest poses: camera-to-world, OpenGL / nerfstudio convention
    (x right, y up, camera looks down -z).
gsplat + COLMAP: OpenCV convention
    (x right, y down, camera looks down +z), world-to-camera "viewmat".

Pixel coordinates follow COLMAP/gsplat: origin at the top-left corner of the
top-left pixel, so pixel centers are at +0.5 and scaling an image by s scales
fx, fy, cx, cy all by s.
"""

import numpy as np

# Right-multiplying a c2w by this flips the camera's own y and z axes.
GL_TO_CV = np.diag([1.0, -1.0, -1.0, 1.0])


def c2w_gl_to_cv(c2w_gl: np.ndarray) -> np.ndarray:
    return c2w_gl @ GL_TO_CV


def c2w_cv_to_gl(c2w_cv: np.ndarray) -> np.ndarray:
    return c2w_cv @ GL_TO_CV


def viewmat_from_c2w_gl(c2w_gl: np.ndarray) -> np.ndarray:
    """World-to-camera (OpenCV) matrix, as gsplat's `viewmats` expects."""
    return np.linalg.inv(c2w_gl_to_cv(c2w_gl))


def intrinsics_matrix(fl_x: float, fl_y: float, cx: float, cy: float) -> np.ndarray:
    return np.array([[fl_x, 0.0, cx], [0.0, fl_y, cy], [0.0, 0.0, 1.0]])


def scale_intrinsics(K: np.ndarray, sx: float, sy: float) -> np.ndarray:
    K = K.copy()
    K[0] *= sx
    K[1] *= sy
    K[2] = [0.0, 0.0, 1.0]
    return K


def qvec_to_rotmat(q) -> np.ndarray:
    """Unit quaternion (w, x, y, z) -> 3x3 rotation matrix."""
    w, x, y, z = np.asarray(q, dtype=np.float64) / np.linalg.norm(q)
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
        [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
        [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)],
    ])


def colmap_pose_to_c2w_gl(qvec, tvec) -> np.ndarray:
    """COLMAP image pose (world-to-camera, OpenCV) -> manifest c2w (OpenGL)."""
    w2c = np.eye(4)
    w2c[:3, :3] = qvec_to_rotmat(qvec)
    w2c[:3, 3] = tvec
    return c2w_cv_to_gl(np.linalg.inv(w2c))


def project(points_world: np.ndarray, viewmat: np.ndarray, K: np.ndarray):
    """Project Nx3 world points. Returns (uv Nx2, depth N); depth>0 = in front."""
    pts = np.asarray(points_world, dtype=np.float64)
    cam = pts @ viewmat[:3, :3].T + viewmat[:3, 3]
    depth = cam[:, 2]
    with np.errstate(divide="ignore", invalid="ignore"):
        uv = (cam[:, :2] / depth[:, None]) * [K[0, 0], K[1, 1]] + [K[0, 2], K[1, 2]]
    return uv, depth


def camera_center(c2w: np.ndarray) -> np.ndarray:
    return c2w[:3, 3]


def camera_forward_gl(c2w_gl: np.ndarray) -> np.ndarray:
    """World-space viewing direction of an OpenGL-convention camera."""
    return -c2w_gl[:3, 2]

In [ ]:
%%writefile /kaggle/working/gs_pipeline/capture_manifest.schema.json
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "$id": "capture_manifest.schema.json",
  "title": "Capture manifest v1",
  "description": "One capture bundle = a folder holding manifest.json, the images it references, and an optional points3d.ply. Compatible with nerfstudio transforms.json. Poses are camera-to-world, OpenGL convention (x right, y up, camera looks down -z). Intrinsics are in pixels of the saved image. Per-frame intrinsics override the top-level defaults.",
  "type": "object",
  "required": ["schema_version", "frames"],
  "properties": {
    "schema_version": { "type": "string", "pattern": "^1\\.\\d+$" },
    "capture_id": { "type": "string" },
    "capture_mode": {
      "description": "object = orbit around one object (uses object_bounds); scene = a whole room captured from several stations (extensions.scene). Absent = object.",
      "enum": ["object", "scene"]
    },
    "source": { "enum": ["arkit", "arcore", "arkit+colmap", "arcore+colmap", "colmap", "synthetic", "other"] },
    "device": { "type": "object" },
    "camera_model": { "enum": ["PINHOLE", "OPENCV"] },
    "fl_x": { "$ref": "#/$defs/positive" },
    "fl_y": { "$ref": "#/$defs/positive" },
    "cx": { "type": "number" },
    "cy": { "type": "number" },
    "w": { "type": "integer", "minimum": 1 },
    "h": { "type": "integer", "minimum": 1 },
    "k1": { "type": "number" },
    "k2": { "type": "number" },
    "p1": { "type": "number" },
    "p2": { "type": "number" },
    "points3d_path": { "type": "string" },
    "object_bounds": {
      "description": "Vertical cylinder enclosing the object: center = footprint center on the supporting surface (world +y is up), radius, height.",
      "type": "object",
      "required": ["center", "radius"],
      "properties": {
        "center": { "$ref": "#/$defs/vec3" },
        "radius": { "$ref": "#/$defs/positive" },
        "height": { "$ref": "#/$defs/positive" }
      }
    },
    "frames": {
      "type": "array",
      "minItems": 1,
      "items": {
        "type": "object",
        "required": ["file_path", "transform_matrix"],
        "properties": {
          "file_path": { "type": "string", "minLength": 1 },
          "transform_matrix": {
            "type": "array",
            "minItems": 4,
            "maxItems": 4,
            "items": { "type": "array", "minItems": 4, "maxItems": 4, "items": { "type": "number" } }
          },
          "fl_x": { "$ref": "#/$defs/positive" },
          "fl_y": { "$ref": "#/$defs/positive" },
          "cx": { "type": "number" },
          "cy": { "type": "number" },
          "w": { "type": "integer", "minimum": 1 },
          "h": { "type": "integer", "minimum": 1 },
          "timestamp": { "type": "number" },
          "tracking_state": { "type": "string" },
          "blur_score": { "type": "number" },
          "angular_velocity": { "type": "number" },
          "exposure_duration": { "type": "number" },
          "iso": { "type": "number" },
          "target_id": { "type": ["integer", "string"] },
          "depth_path": { "type": "string" }
        }
      }
    },
    "extensions": { "type": "object" }
  },
  "$defs": {
    "positive": { "type": "number", "exclusiveMinimum": 0 },
    "vec3": { "type": "array", "minItems": 3, "maxItems": 3, "items": { "type": "number" } }
  }
}

In [ ]:
%%writefile /kaggle/working/gs_pipeline/colmap_io.py
"""Read COLMAP binary models and convert them into a capture manifest.

Used for public reference datasets (GPU golden test) and later as the output
step of the GLOMAP/COLMAP fallback path for unposed footage.
"""

import json
import struct
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from . import SCHEMA_VERSION
from .cameras import colmap_pose_to_c2w_gl
from .ply_io import write_points_ply

# model_id -> (name, num_params)
CAMERA_MODELS = {
    0: ("SIMPLE_PINHOLE", 3), 1: ("PINHOLE", 4), 2: ("SIMPLE_RADIAL", 4), 3: ("RADIAL", 5),
    4: ("OPENCV", 8), 5: ("OPENCV_FISHEYE", 8), 6: ("FULL_OPENCV", 12),
}


@dataclass
class ColmapCamera:
    id: int
    model: str
    width: int
    height: int
    params: np.ndarray

    def pinhole(self) -> tuple[float, float, float, float]:
        """(fx, fy, cx, cy). Only distortion-free models are accepted."""
        p = self.params
        if self.model == "SIMPLE_PINHOLE":
            return p[0], p[0], p[1], p[2]
        if self.model == "PINHOLE":
            return p[0], p[1], p[2], p[3]
        raise ValueError(f"camera {self.id}: model {self.model} has distortion; undistort the images first")


@dataclass
class ColmapImage:
    id: int
    qvec: np.ndarray
    tvec: np.ndarray
    camera_id: int
    name: str
    xys: np.ndarray          # Nx2 keypoints, pixels of the original camera
    point3d_ids: np.ndarray  # N, -1 = untriangulated


def _read(fh, fmt):
    return struct.unpack("<" + fmt, fh.read(struct.calcsize("<" + fmt)))


def read_cameras_bin(path) -> dict[int, ColmapCamera]:
    cams = {}
    with open(path, "rb") as fh:
        (n,) = _read(fh, "Q")
        for _ in range(n):
            cam_id, model_id, w, h = _read(fh, "iiQQ")
            name, nparams = CAMERA_MODELS[model_id]
            cams[cam_id] = ColmapCamera(cam_id, name, w, h, np.array(_read(fh, "d" * nparams)))
    return cams


def read_images_bin(path) -> dict[int, ColmapImage]:
    images = {}
    with open(path, "rb") as fh:
        (n,) = _read(fh, "Q")
        for _ in range(n):
            img_id, *pose, cam_id = _read(fh, "idddddddi")
            name = b""
            while (c := fh.read(1)) != b"\x00":
                name += c
            (npts,) = _read(fh, "Q")
            pts = np.frombuffer(fh.read(24 * npts), dtype=[("x", "<f8"), ("y", "<f8"), ("id", "<i8")])
            images[img_id] = ColmapImage(
                img_id, np.array(pose[:4]), np.array(pose[4:]), cam_id, name.decode(),
                np.stack([pts["x"], pts["y"]], axis=1), pts["id"].copy(),
            )
    return images


def read_points3d_bin(path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Returns (ids N, xyz Nx3, rgb Nx3 uint8)."""
    data = Path(path).read_bytes()
    (n,) = struct.unpack_from("<Q", data, 0)
    off = 8
    ids, xyz, rgb = np.empty(n, np.int64), np.empty((n, 3)), np.empty((n, 3), np.uint8)
    head = struct.Struct("<QdddBBBdQ")
    for i in range(n):
        pid, x, y, z, r, g, b, _err, track_len = head.unpack_from(data, off)
        ids[i], xyz[i], rgb[i] = pid, (x, y, z), (r, g, b)
        off += head.size + 8 * track_len
    return ids, xyz, rgb


def convert_colmap(scene_dir, images_subdir: str = "images", sparse_subdir: str = "sparse/0",
                   manifest_name: str = "manifest.json") -> Path:
    """Write <scene_dir>/manifest.json (+ points3d.ply) from a COLMAP model.

    Intrinsics are rescaled to the actual size of the images in `images_subdir`
    (e.g. mip-NeRF 360's `images_4` are 4x downscaled).
    """
    from PIL import Image

    scene_dir = Path(scene_dir)
    sparse = scene_dir / sparse_subdir
    cams = read_cameras_bin(sparse / "cameras.bin")
    images = read_images_bin(sparse / "images.bin")

    frames = []
    for img in sorted(images.values(), key=lambda im: im.name):
        rel = f"{images_subdir}/{img.name}"
        path = scene_dir / rel
        if not path.exists():
            continue
        cam = cams[img.camera_id]
        with Image.open(path) as im:
            w, h = im.size
        sx, sy = w / cam.width, h / cam.height
        fx, fy, cx, cy = cam.pinhole()
        frames.append({
            "file_path": rel,
            "transform_matrix": colmap_pose_to_c2w_gl(img.qvec, img.tvec).tolist(),
            "fl_x": fx * sx, "fl_y": fy * sy, "cx": cx * sx, "cy": cy * sy, "w": w, "h": h,
        })
    if not frames:
        raise FileNotFoundError(f"no images from the COLMAP model found under {scene_dir / images_subdir}")

    manifest = {
        "schema_version": SCHEMA_VERSION,
        "capture_id": scene_dir.name,
        "source": "colmap",
        "camera_model": "PINHOLE",
        "frames": frames,
    }
    points_file = sparse / "points3D.bin"
    if points_file.exists():
        _, xyz, rgb = read_points3d_bin(points_file)
        write_points_ply(scene_dir / "points3d.ply", xyz, rgb)
        manifest["points3d_path"] = "points3d.ply"

    out = scene_dir / manifest_name
    out.write_text(json.dumps(manifest, indent=1))
    return out

In [ ]:
%%writefile /kaggle/working/gs_pipeline/coverage.py
"""Hole report for scene (room) splats: where can a visitor see through the room?

From every capture spot (extensions.scene.stations) a 90-degree cubemap is rendered at standing eye
height. Pixels whose accumulated opacity stays below HOLE_ALPHA see through the splat (in the viewer the
grid or background shows there). The hole fraction is the room-quality KPI, and the images (holes painted
magenta) show the user what to re-shoot.
"""

import numpy as np

from .cameras import intrinsics_matrix
from .scene import scene_info

HOLE_ALPHA = 0.5
EYE_HEIGHT_M = 1.2
FACE_PX = 256

# Cubemap faces as (forward, down) in world axes (y up). OpenCV camera: x right, y down, z forward.
FACES = {
    "front": ((0, 0, -1), (0, -1, 0)), "right": ((1, 0, 0), (0, -1, 0)),
    "back": ((0, 0, 1), (0, -1, 0)), "left": ((-1, 0, 0), (0, -1, 0)),
    "up": ((0, 1, 0), (0, 0, 1)), "down": ((0, -1, 0), (0, 0, -1)),
}


def face_viewmat(eye, forward, down) -> np.ndarray:
    """World-to-camera (OpenCV) for a camera at eye looking along forward with image-down = down."""
    z = np.asarray(forward, float)
    y = np.asarray(down, float)
    x = np.cross(y, z)
    R = np.stack([x, y, z])  # rows = camera axes in world coords
    vm = np.eye(4)
    vm[:3, :3] = R
    vm[:3, 3] = -R @ np.asarray(eye, float)
    return vm


def cubemap_viewmats(eye) -> list[np.ndarray]:
    return [face_viewmat(eye, f, d) for f, d in FACES.values()]


def face_K(size: int = FACE_PX) -> np.ndarray:
    return intrinsics_matrix(size / 2, size / 2, size / 2, size / 2)  # 90 degree field of view


def snap_to_cameras(spots_xz: np.ndarray, cameras_xz: np.ndarray, iters: int = 10, max_move: float = 1.0) -> np.ndarray:
    """Move recorded spot centres onto the camera clusters they belong to (k-means seeded at the spots).
    The app records spots in the AR world, which drifted up to ~2 m on the first room, while the cameras
    are COLMAP-refined; a drifted spot can sit inside furniture or a wall and see "holes" that aren't there.
    A spot that would move more than max_move keeps its recorded position: its own photos are missing (on the
    first room COLMAP dropped all of spot 4's) and the nearest cluster belongs to something else."""
    c = spots_xz.astype(float).copy()
    if len(cameras_xz) == 0 or len(c) == 0:
        return c
    for _ in range(iters):
        nearest = np.argmin(np.linalg.norm(cameras_xz[:, None] - c[None], axis=-1), axis=1)
        for k in range(len(c)):
            members = cameras_xz[nearest == k]
            if len(members):
                c[k] = np.median(members, axis=0)
    far = np.linalg.norm(c - spots_xz, axis=1) > max_move
    c[far] = spots_xz[far]
    return c


def station_eyes(manifest: dict, floor_y: float, camera_centers: np.ndarray | None = None) -> list[np.ndarray]:
    """Standing eye positions above each capture spot (training coordinates)."""
    spots = np.array([[s["center"][0], s["center"][2]] for s in scene_info(manifest).get("stations", [])
                      if "center" in s], float).reshape(-1, 2)
    if camera_centers is not None and len(spots):
        spots = snap_to_cameras(spots, np.asarray(camera_centers, float)[:, [0, 2]])
    return [np.array([x, floor_y + EYE_HEIGHT_M, z], float) for x, z in spots]


def hole_fraction(alpha: np.ndarray) -> float:
    return float((np.asarray(alpha) < HOLE_ALPHA).mean())


def paint_holes(rgb: np.ndarray, alpha: np.ndarray) -> np.ndarray:
    """uint8 image: the render, with see-through pixels painted magenta."""
    img = (np.clip(np.nan_to_num(rgb), 0, 1) * 255).astype(np.uint8)
    img[np.asarray(alpha) < HOLE_ALPHA] = (255, 0, 255)
    return img


def cubemap_sheet(faces: list[np.ndarray]) -> np.ndarray:
    """6 faces (front right back left / up down) -> one 4x2 image sheet."""
    s = faces[0].shape[0]
    sheet = np.zeros((2 * s, 4 * s, 3), np.uint8)
    for k, f in enumerate(faces):
        r, c = divmod(k, 4)
        sheet[r * s:(r + 1) * s, c * s:(c + 1) * s] = f
    return sheet


def hole_report(render_fn, manifest: dict, floor_y: float, out_dir, camera_centers=None) -> dict:
    """render_fn(viewmat 4x4, K 3x3, w, h) -> (rgb HxWx3 float, alpha HxW). Saves holes_station<k>.png."""
    from PIL import Image

    eyes = station_eyes(manifest, floor_y, camera_centers)
    per_station, alphas = [], []
    K = face_K()
    for k, eye in enumerate(eyes):
        faces = []
        station_alpha = []
        for vm in cubemap_viewmats(eye):
            rgb, alpha = render_fn(vm, K, FACE_PX, FACE_PX)
            station_alpha.append(alpha)
            faces.append(paint_holes(rgb, alpha))
        Image.fromarray(cubemap_sheet(faces)).save(f"{out_dir}/holes_station{k}.png")
        per_station.append(round(hole_fraction(np.stack(station_alpha)), 4))
        alphas += station_alpha
    return {"hole_fraction": round(hole_fraction(np.stack(alphas)), 4) if alphas else None,
            "hole_fraction_per_station": per_station, "eye_height_m": EYE_HEIGHT_M,
            "eyes": [[round(float(v), 3) for v in e] for e in eyes]}

In [ ]:
%%writefile /kaggle/working/gs_pipeline/depth.py
"""Monocular depth priors for scene (room) captures.

Blank ceilings and walls give COLMAP almost no points (the first room: 38.8k points, nearly none on the
ceiling), so 3DGS gets no Gaussians there and the room has holes, or it fakes them with needle-shaped
"fur". A monocular depth network still sees those surfaces. Its output is relative (unknown scale and
shift per image), so each map is aligned to the metric sparse points that project into that image.
The aligned maps are used twice:
1. dense initialisation: back-projected, multi-view-consistent points fill the blank surfaces;
2. a depth loss during training (train.py), so those surfaces stay flat instead of turning to fur.

    python -m gs_pipeline depth <bundle>/manifest.json --out <depth_dir>

writes <depth_dir>/<frame>.npy (float16 relative disparity), align.json and points3d_dense.ply.
"""

import json
from pathlib import Path

import numpy as np

from .cameras import project, scale_intrinsics, viewmat_from_c2w_gl
from .manifest import load_frames
from .validate import load_points

MODEL = "depth-anything/Depth-Anything-V2-Large-hf"
DEPTH_SIDE = 518          # long side of the saved depth maps (the network's native resolution)
MIN_INLIERS = 30          # fewer aligned sparse points than this: no prior for the frame
ALIGN_FILE = "align.json"
DENSE_FILE = "points3d_dense.ply"


# ---------------------------------------------------------------- pure helpers (unit-tested)

def depth_size(width: int, height: int, side: int = DEPTH_SIDE) -> tuple[int, int]:
    """(w, h) of a frame's depth map: long side = `side`, same aspect."""
    s = side / max(width, height)
    return max(1, round(width * s)), max(1, round(height * s))


def sparse_samples(points: np.ndarray, viewmat: np.ndarray, K: np.ndarray, w: int, h: int,
                   min_depth: float = 0.05) -> tuple[np.ndarray, np.ndarray]:
    """Integer pixel (col, row) Nx2 and depth N of the points that land inside a w x h image."""
    uv, z = project(points, viewmat, K)
    ok = (z > min_depth) & np.isfinite(uv).all(1) & (uv[:, 0] >= 0) & (uv[:, 0] < w) & (uv[:, 1] >= 0) & (uv[:, 1] < h)
    return np.floor(uv[ok]).astype(int), z[ok]


def fit_disparity(mono: np.ndarray, z: np.ndarray, iters: int = 300, rel_tol: float = 0.08,
                  seed: int = 0) -> tuple[float, float, np.ndarray]:
    """Robust fit of 1/z = a * mono + b (a > 0). Points behind occluders (the sparse cloud has no
    visibility info) are simply outliers. Returns (a, b, inlier mask); a = 0 if no fit was found."""
    target = 1.0 / z
    n = len(mono)
    best = np.zeros(n, bool)
    if n < 2:
        return 0.0, 0.0, best
    rng = np.random.default_rng(seed)
    for _ in range(iters):
        i, j = rng.choice(n, 2, replace=False)
        if abs(mono[i] - mono[j]) < 1e-6:
            continue
        a = (target[i] - target[j]) / (mono[i] - mono[j])
        if a <= 0:
            continue
        b = target[i] - a * mono[i]
        inl = np.abs(a * mono + b - target) < rel_tol * target
        if inl.sum() > best.sum():
            best = inl
    if best.sum() < 2:
        return 0.0, 0.0, best
    A = np.stack([mono[best], np.ones(best.sum())], 1)
    a, b = np.linalg.lstsq(A, target[best], rcond=None)[0]
    if a <= 0:
        return 0.0, 0.0, np.zeros(n, bool)
    inl = np.abs(a * mono + b - target) < rel_tol * target
    return float(a), float(b), inl


def metric_depth(mono: np.ndarray, a: float, b: float, max_depth: float = 30.0) -> np.ndarray:
    """Aligned metric depth (m); 0 where the aligned disparity is non-positive or the depth is absurd."""
    disp = a * mono.astype(np.float64) + b
    with np.errstate(divide="ignore"):
        d = np.where(disp > 1.0 / max_depth, 1.0 / disp, 0.0)
    return d


def backproject(depth: np.ndarray, K: np.ndarray, c2w_gl: np.ndarray, stride: int) -> tuple[np.ndarray, np.ndarray]:
    """World points (Nx3) of every `stride`-th pixel with depth > 0, and their (row, col) pixels."""
    rows, cols = np.mgrid[stride // 2:depth.shape[0]:stride, stride // 2:depth.shape[1]:stride]
    rows, cols = rows.ravel(), cols.ravel()
    z = depth[rows, cols]
    ok = z > 0
    rows, cols, z = rows[ok], cols[ok], z[ok]
    x = (cols + 0.5 - K[0, 2]) / K[0, 0] * z
    y = (rows + 0.5 - K[1, 2]) / K[1, 1] * z
    cam = np.stack([x, y, z], 1)                       # OpenCV camera frame
    c2w_cv = np.linalg.inv(viewmat_from_c2w_gl(c2w_gl))
    return cam @ c2w_cv[:3, :3].T + c2w_cv[:3, 3], np.stack([rows, cols], 1)


def consistent_views(points: np.ndarray, depths: list[np.ndarray], Ks: list[np.ndarray], viewmats: list[np.ndarray],
                     skip: int, rel_tol: float = 0.05) -> np.ndarray:
    """For each point, the number of other views whose depth map agrees with it within rel_tol."""
    count = np.zeros(len(points), int)
    for k, (d, K, vm) in enumerate(zip(depths, Ks, viewmats)):
        if k == skip or d is None:
            continue
        h, w = d.shape
        uv, z = project(points, vm, K)
        ok = (z > 0.05) & np.isfinite(uv).all(1) & (uv[:, 0] >= 0) & (uv[:, 0] < w) & (uv[:, 1] >= 0) & (uv[:, 1] < h)
        idx = np.nonzero(ok)[0]
        dz = d[np.floor(uv[idx, 1]).astype(int), np.floor(uv[idx, 0]).astype(int)]
        agree = (dz > 0) & (np.abs(z[idx] - dz) < rel_tol * dz)
        count[idx[agree]] += 1
    return count


def voxel_merge(xyz: np.ndarray, rgb: np.ndarray, voxel: float) -> tuple[np.ndarray, np.ndarray]:
    """One point per occupied voxel: the mean position and colour of the points inside it."""
    if len(xyz) == 0:
        return xyz, rgb
    keys = np.floor(xyz / voxel).astype(np.int64)
    _, inv, cnt = np.unique(keys, axis=0, return_inverse=True, return_counts=True)
    inv = inv.ravel()
    out_xyz = np.zeros((len(cnt), 3))
    out_rgb = np.zeros((len(cnt), 3))
    np.add.at(out_xyz, inv, xyz)
    np.add.at(out_rgb, inv, rgb.astype(np.float64))
    return (out_xyz / cnt[:, None]).astype(np.float32), np.round(out_rgb / cnt[:, None]).astype(np.uint8)


def dense_points(frames, depths: list[np.ndarray | None], colors: list[np.ndarray], stride: int = 8,
                 rel_tol: float = 0.05, min_agree: int = 2, voxel: float = 0.02) -> tuple[np.ndarray, np.ndarray]:
    """Fused, multi-view-consistent points from aligned depth maps (all at depth-map resolution)."""
    Ks = [None if d is None else scale_intrinsics(f.K, d.shape[1] / f.width, d.shape[0] / f.height)
          for f, d in zip(frames, depths)]
    vms = [viewmat_from_c2w_gl(f.c2w_gl) for f in frames]
    all_xyz, all_rgb = [], []
    for i, (f, d) in enumerate(zip(frames, depths)):
        if d is None:
            continue
        xyz, px = backproject(d, Ks[i], f.c2w_gl, stride)
        keep = consistent_views(xyz, depths, Ks, vms, skip=i, rel_tol=rel_tol) >= min_agree
        all_xyz.append(xyz[keep])
        all_rgb.append(colors[i][px[keep, 0], px[keep, 1]])
    if not all_xyz:
        return np.zeros((0, 3), np.float32), np.zeros((0, 3), np.uint8)
    return voxel_merge(np.concatenate(all_xyz), np.concatenate(all_rgb), voxel)


# ---------------------------------------------------------------- loading (used by train.py)

def frame_key(frame) -> str:
    return Path(frame.image_path).stem


def load_aligned(depth_dir, frames) -> list[np.ndarray | None]:
    """Aligned metric depth per frame at depth-map resolution (None where alignment failed)."""
    depth_dir = Path(depth_dir)
    align = json.loads((depth_dir / ALIGN_FILE).read_text())["frames"]
    out = []
    for f in frames:
        a = align.get(frame_key(f))
        path = depth_dir / f"{frame_key(f)}.npy"
        if not a or a["a"] <= 0 or a["inliers"] < MIN_INLIERS or not path.exists():
            out.append(None)
            continue
        out.append(metric_depth(np.load(path).astype(np.float32), a["a"], a["b"]))
    return out


def has_depth(depth_dir) -> bool:
    return depth_dir is not None and (Path(depth_dir) / ALIGN_FILE).exists()


# ---------------------------------------------------------------- pipeline (GPU for the network)

def predict_mono(frames, model_name: str = MODEL, device: str | None = None) -> list[np.ndarray]:
    """Relative disparity per frame at depth_size(), from Depth-Anything-V2 (bigger = nearer)."""
    import torch
    import torch.nn.functional as F
    from PIL import Image
    from transformers import AutoImageProcessor, AutoModelForDepthEstimation

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModelForDepthEstimation.from_pretrained(model_name).to(device).eval()
    out = []
    for k, f in enumerate(frames):
        with Image.open(f.image_path) as im:
            inputs = processor(images=im.convert("RGB"), return_tensors="pt").to(device)
        with torch.no_grad():
            pred = model(**inputs).predicted_depth  # (1, h', w') relative disparity
        w, h = depth_size(f.width, f.height)
        pred = F.interpolate(pred[:, None].float(), size=(h, w), mode="bilinear", align_corners=False)[0, 0]
        out.append(pred.cpu().numpy())
        if k % 25 == 0:
            print(f"[depth] {k + 1}/{len(frames)} {f.image_path.name}", flush=True)
    return out


def compute_depth(manifest_path, out_dir=None, model_name: str = MODEL, stride: int = 8) -> dict:
    """Predict, align and fuse. Returns the report (also saved inside align.json)."""
    from PIL import Image

    from .ply_io import write_points_ply

    frames, manifest = load_frames(manifest_path)
    out = Path(out_dir) if out_dir else Path(manifest_path).parent / "depth"
    out.mkdir(parents=True, exist_ok=True)
    points = load_points(manifest_path, manifest)
    if points is None or len(points) < MIN_INLIERS:
        raise ValueError("depth priors need the refined sparse points (points3d.ply); run `refine` first")

    monos = predict_mono(frames, model_name)
    align, depths, colors = {}, [], []
    for f, mono in zip(frames, monos):
        np.save(out / f"{frame_key(f)}.npy", mono.astype(np.float16))
        h, w = mono.shape
        K = scale_intrinsics(f.K, w / f.width, h / f.height)
        px, z = sparse_samples(points, viewmat_from_c2w_gl(f.c2w_gl), K, w, h)
        a, b, inl = fit_disparity(mono[px[:, 1], px[:, 0]].astype(np.float64), z)
        rel = float(np.median(np.abs(1 / (a * mono[px[inl, 1], px[inl, 0]] + b) - z[inl]) / z[inl])) if inl.any() else None
        align[frame_key(f)] = {"a": a, "b": b, "points": int(len(z)), "inliers": int(inl.sum()),
                               "median_rel_err": None if rel is None else round(rel, 4)}
        ok = a > 0 and inl.sum() >= MIN_INLIERS
        depths.append(metric_depth(mono, a, b) if ok else None)
        with Image.open(f.image_path) as im:
            colors.append(np.asarray(im.convert("RGB").resize((w, h), Image.BILINEAR)))

    xyz, rgb = dense_points(frames, depths, colors, stride=stride)
    write_points_ply(out / DENSE_FILE, xyz, rgb)
    aligned = sum(d is not None for d in depths)
    errs = [v["median_rel_err"] for v in align.values() if v["median_rel_err"] is not None]
    report = {"model": model_name, "frames": len(frames), "aligned": aligned,
              "median_rel_err": round(float(np.median(errs)), 4) if errs else None,
              "dense_points": int(len(xyz)), "sparse_points": int(len(points)), "dir": str(out)}
    (out / ALIGN_FILE).write_text(json.dumps({"report": report, "frames": align}, indent=1))
    return report

In [ ]:
%%writefile /kaggle/working/gs_pipeline/manifest.py
"""Load capture manifests into per-frame camera records."""

import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from .cameras import intrinsics_matrix

SCHEMA_PATH = Path(__file__).with_name("capture_manifest.schema.json")
INTRINSIC_KEYS = ("fl_x", "fl_y", "cx", "cy", "w", "h")


@dataclass
class CameraFrame:
    image_path: Path
    K: np.ndarray        # 3x3, pixels of the saved image
    width: int
    height: int
    c2w_gl: np.ndarray   # 4x4 camera-to-world, OpenGL convention
    index: int


def load_schema() -> dict:
    return json.loads(SCHEMA_PATH.read_text())


def schema_errors(manifest: dict) -> list[str]:
    import jsonschema

    validator = jsonschema.Draft202012Validator(load_schema())
    return [
        f"{'/'.join(map(str, e.absolute_path)) or '<root>'}: {e.message}"
        for e in sorted(validator.iter_errors(manifest), key=lambda e: list(e.absolute_path))
    ]


def read_manifest(path) -> dict:
    return json.loads(Path(path).read_text())


def frame_intrinsics(manifest: dict, frame: dict) -> dict:
    """Per-frame intrinsics with top-level fallbacks; missing keys are omitted."""
    out = {}
    for k in INTRINSIC_KEYS:
        v = frame.get(k, manifest.get(k))
        if v is not None:
            out[k] = v
    return out


def load_frames(manifest_path) -> tuple[list[CameraFrame], dict]:
    manifest_path = Path(manifest_path)
    manifest = read_manifest(manifest_path)
    frames = []
    for i, f in enumerate(manifest["frames"]):
        intr = frame_intrinsics(manifest, f)
        missing = [k for k in INTRINSIC_KEYS if k not in intr]
        if missing:
            raise ValueError(f"frame {i} ({f['file_path']}) missing intrinsics: {missing}")
        frames.append(CameraFrame(
            image_path=manifest_path.parent / f["file_path"],
            K=intrinsics_matrix(intr["fl_x"], intr["fl_y"], intr["cx"], intr["cy"]),
            width=int(intr["w"]),
            height=int(intr["h"]),
            c2w_gl=np.asarray(f["transform_matrix"], dtype=np.float64),
            index=i,
        ))
    return frames, manifest


def points_path(manifest_path, manifest: dict) -> Path | None:
    rel = manifest.get("points3d_path")
    return Path(manifest_path).parent / rel if rel else None

In [ ]:
%%writefile /kaggle/working/gs_pipeline/overlay.py
"""Draw the AR feature points and the object-bounds cylinder onto captured photos via the manifest poses.

If image orientation or pose conventions are wrong (e.g. a portrait/landscape or mirror mix-up on a
new phone), the drawn geometry visibly misses the object, so this catches it on CPU, before any GPU run.
"""

from pathlib import Path

import numpy as np

from .cameras import project, viewmat_from_c2w_gl
from .manifest import load_frames, points_path
from .ply_io import read_points_ply
from .bounds import refine_bounds, user_bounds


def _cylinder_rings(center, radius, height, n=48):
    """Bottom + top circles of the object bounds (center = footprint center, y up)."""
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    ring = np.stack([radius * np.cos(t), np.zeros(n), radius * np.sin(t)], axis=1)
    base = np.asarray(center, float) + ring
    return [base, base + [0.0, height, 0.0]]


def overlay(manifest_path, out_png, num_frames: int = 6, max_points: int = 4000) -> Path:
    from PIL import Image, ImageDraw

    frames, manifest = load_frames(manifest_path)
    pts = None
    pp = points_path(manifest_path, manifest)
    if pp is not None and pp.exists():
        pts, _ = read_points_ply(pp)
        if len(pts) > max_points:
            pts = pts[np.random.default_rng(0).choice(len(pts), max_points, replace=False)]
    rings = []  # (points, color): user-tapped cylinder in cyan, point-refined cylinder in green
    user = user_bounds(manifest)
    if user is not None:
        rings += [(r, (40, 220, 255)) for r in _cylinder_rings(user.base, user.radius, user.height)]
        all_pts = read_points_ply(pp)[0] if pp is not None and pp.exists() else None
        refined = refine_bounds(manifest, all_pts)
        if refined is not None and refined.source != user.source:
            rings += [(r, (60, 255, 60)) for r in _cylinder_rings(refined.base, refined.radius, refined.height)]

    picks = np.linspace(0, len(frames) - 1, min(num_frames, len(frames))).round().astype(int)
    tiles = []
    for i in picks:
        fr = frames[i]
        im = Image.open(fr.image_path).convert("RGB")
        draw = ImageDraw.Draw(im)
        vm = viewmat_from_c2w_gl(fr.c2w_gl)
        if pts is not None:
            uv, depth = project(pts, vm, fr.K)
            for (u, v), d in zip(uv, depth):
                if d > 0 and 0 <= u < fr.width and 0 <= v < fr.height:
                    draw.ellipse([u - 2, v - 2, u + 2, v + 2], fill=(255, 40, 40))
        for ring, color in rings:
            uv, depth = project(ring, vm, fr.K)
            if (depth > 0).all():
                draw.line([tuple(p) for p in uv] + [tuple(uv[0])], fill=color, width=3)
        draw.text((10, 10), f"frame {i}: {fr.image_path.name}", fill=(255, 255, 0))
        im.thumbnail((640, 640))
        tiles.append(im)

    cols = 3
    w, h = max(t.width for t in tiles), max(t.height for t in tiles)
    sheet = Image.new("RGB", (cols * w, ((len(tiles) + cols - 1) // cols) * h), (30, 30, 30))
    for k, t in enumerate(tiles):
        sheet.paste(t, ((k % cols) * w, (k // cols) * h))
    out_png = Path(out_png)
    sheet.save(out_png)
    return out_png

In [ ]:
%%writefile /kaggle/working/gs_pipeline/ply_io.py
"""Binary little-endian PLY I/O for point clouds and 3DGS splats (no extra deps).

Splat files use the standard INRIA 3DGS layout that SuperSplat, Unity/Unreal
splat plugins and most viewers read.
"""

import numpy as np

_PLY_TYPES = {
    "float": "<f4", "float32": "<f4", "double": "<f8", "float64": "<f8",
    "uchar": "u1", "uint8": "u1", "char": "i1", "int8": "i1",
    "ushort": "<u2", "uint16": "<u2", "short": "<i2", "int16": "<i2",
    "uint": "<u4", "uint32": "<u4", "int": "<i4", "int32": "<i4",
}
_NP_TO_PLY = {"<f4": "float", "<f8": "double", "|u1": "uchar", "<i4": "int"}


def write_ply(path, data: np.ndarray) -> None:
    """Write a structured numpy array as a single 'vertex' element."""
    header = ["ply", "format binary_little_endian 1.0", f"element vertex {len(data)}"]
    header += [f"property {_NP_TO_PLY[data.dtype[name].str]} {name}" for name in data.dtype.names]
    header.append("end_header")
    with open(path, "wb") as fh:
        fh.write(("\n".join(header) + "\n").encode("ascii"))
        fh.write(data.tobytes())


def read_ply(path) -> np.ndarray:
    """Read the 'vertex' element of a binary little-endian PLY (must be the first element)."""
    with open(path, "rb") as fh:
        if fh.readline().strip() != b"ply":
            raise ValueError(f"{path}: not a PLY file")
        fmt, count, fields, current = None, 0, [], None
        while True:
            line = fh.readline().decode("ascii").strip()
            if line == "end_header":
                break
            parts = line.split()
            if not parts:
                continue
            if parts[0] == "format":
                fmt = parts[1]
            elif parts[0] == "element":
                current = parts[1]
                if current == "vertex":
                    count = int(parts[2])
            elif parts[0] == "property" and current == "vertex":
                if parts[1] == "list":
                    raise ValueError(f"{path}: list properties on vertices are not supported")
                fields.append((parts[2], _PLY_TYPES[parts[1]]))
        if fmt != "binary_little_endian":
            raise ValueError(f"{path}: only binary_little_endian PLY is supported (got {fmt})")
        dtype = np.dtype(fields)
        return np.frombuffer(fh.read(dtype.itemsize * count), dtype=dtype, count=count)


def write_points_ply(path, xyz: np.ndarray, rgb: np.ndarray | None = None) -> None:
    fields = [("x", "<f4"), ("y", "<f4"), ("z", "<f4")]
    if rgb is not None:
        fields += [("red", "u1"), ("green", "u1"), ("blue", "u1")]
    data = np.empty(len(xyz), dtype=fields)
    xyz = np.asarray(xyz, dtype=np.float32)
    data["x"], data["y"], data["z"] = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    if rgb is not None:
        rgb = np.asarray(rgb, dtype=np.uint8)
        data["red"], data["green"], data["blue"] = rgb[:, 0], rgb[:, 1], rgb[:, 2]
    write_ply(path, data)


def read_points_ply(path) -> tuple[np.ndarray, np.ndarray | None]:
    """Returns (xyz Nx3 float32, rgb Nx3 uint8 or None). Non-finite points (ARCore emits some) are dropped."""
    v = read_ply(path)
    xyz = np.stack([v["x"], v["y"], v["z"]], axis=1).astype(np.float32)
    rgb = None
    if {"red", "green", "blue"} <= set(v.dtype.names):
        rgb = np.stack([v["red"], v["green"], v["blue"]], axis=1).astype(np.uint8)
    ok = np.isfinite(xyz).all(axis=1)
    if not ok.all():
        xyz, rgb = xyz[ok], (rgb[ok] if rgb is not None else None)
    return xyz, rgb


def write_gaussians_ply(path, means, scales_log, quats_wxyz, opacities_logit, sh0, shN) -> None:
    """INRIA 3DGS layout. sh0: (N,3), shN: (N,K,3) with K = (deg+1)^2 - 1.

    Values are stored pre-activation (log scales, logit opacity), as viewers expect.
    """
    n = len(means)
    shN = np.asarray(shN, dtype=np.float32)
    sh_rest = shN.transpose(0, 2, 1).reshape(n, shN.shape[1] * 3)  # channel-major; also valid for n == 0
    names = ["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1", "f_dc_2"]
    names += [f"f_rest_{i}" for i in range(sh_rest.shape[1])]
    names += ["opacity", "scale_0", "scale_1", "scale_2", "rot_0", "rot_1", "rot_2", "rot_3"]
    cols = np.concatenate([
        np.asarray(means, np.float32), np.zeros((n, 3), np.float32), np.asarray(sh0, np.float32), sh_rest,
        np.asarray(opacities_logit, np.float32).reshape(n, 1), np.asarray(scales_log, np.float32),
        np.asarray(quats_wxyz, np.float32),
    ], axis=1)
    data = np.empty(n, dtype=[(name, "<f4") for name in names])
    for i, name in enumerate(names):
        data[name] = cols[:, i]
    write_ply(path, data)


def read_gaussians_ply(path) -> dict:
    v = read_ply(path)
    rest = sorted((n for n in v.dtype.names if n.startswith("f_rest_")), key=lambda s: int(s[7:]))
    n = len(v)

    def col(names):
        return np.stack([v[k] for k in names], axis=1)

    return {
        "means": col(["x", "y", "z"]),
        "sh0": col(["f_dc_0", "f_dc_1", "f_dc_2"]),
        "shN": (col(rest).reshape(n, 3, len(rest) // 3).transpose(0, 2, 1) if rest
                else np.zeros((n, 0, 3), np.float32)),
        "opacities_logit": v["opacity"].copy(),
        "scales_log": col(["scale_0", "scale_1", "scale_2"]),
        "quats_wxyz": col(["rot_0", "rot_1", "rot_2", "rot_3"]),
    }

In [ ]:
%%writefile /kaggle/working/gs_pipeline/posecheck.py
"""Measure how consistent a bundle's poses + intrinsics are with its own photos (CPU, needs OpenCV).

SIFT matches between photo pairs that look at the object from similar directions are filtered with a
pose-independent RANSAC, then scored by their Sampson (epipolar) distance under the manifest's poses.
Sharp splats need roughly <= 1-2 px; the first ARCore-only captures measured 10-37 px.
"""

import numpy as np

from .cameras import c2w_gl_to_cv
from .manifest import load_frames


def _sampson_px(Ki, Kj, ci, cj, p1, p2):
    A, B = c2w_gl_to_cv(ci), c2w_gl_to_cv(cj)
    rel = np.linalg.inv(B) @ A
    R, t = rel[:3, :3], rel[:3, 3]
    tx = np.array([[0, -t[2], t[1]], [t[2], 0, -t[0]], [-t[1], t[0], 0]])
    F = np.linalg.inv(Kj).T @ tx @ R @ np.linalg.inv(Ki)
    x1, x2 = np.c_[p1, np.ones(len(p1))], np.c_[p2, np.ones(len(p2))]
    Fx1, Ftx2 = x1 @ F.T, x2 @ F
    return np.sqrt(np.einsum("ij,ij->i", x2, Fx1) ** 2 /
                   (Fx1[:, 0] ** 2 + Fx1[:, 1] ** 2 + Ftx2[:, 0] ** 2 + Ftx2[:, 1] ** 2))


def posecheck(manifest_path, max_angle_deg: float = 25.0, max_pairs: int = 250, min_inliers: int = 30) -> dict:
    import cv2

    frames, _ = load_frames(manifest_path)
    sift = cv2.SIFT_create(4000)
    feats = [sift.detectAndCompute(cv2.imread(str(f.image_path), cv2.IMREAD_GRAYSCALE), None) for f in frames]
    bf = cv2.BFMatcher()
    dirs = np.array([f.c2w_gl[:3, 2] for f in frames])
    n = len(frames)
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)
             if dirs[i] @ dirs[j] > np.cos(np.radians(max_angle_deg))]
    if len(pairs) > max_pairs:
        pairs = [pairs[k] for k in np.linspace(0, len(pairs) - 1, max_pairs).astype(int)]

    errors, gaps = [], []
    for i, j in pairs:
        (k1, d1), (k2, d2) = feats[i], feats[j]
        if d1 is None or d2 is None:
            continue
        ms = [m for m, q in bf.knnMatch(d1, d2, k=2) if m.distance < 0.75 * q.distance]
        if len(ms) < min_inliers:
            continue
        p1 = np.float64([k1[m.queryIdx].pt for m in ms])
        p2 = np.float64([k2[m.trainIdx].pt for m in ms])
        _, mask = cv2.findFundamentalMat(p1, p2, cv2.FM_RANSAC, 2.0, 0.999)
        if mask is None or mask.sum() < min_inliers:
            continue
        keep = mask.ravel() == 1
        e = _sampson_px(frames[i].K, frames[j].K, frames[i].c2w_gl, frames[j].c2w_gl, p1[keep], p2[keep])
        errors.append(float(np.median(e)))
        gaps.append(j - i)

    errors, gaps = np.array(errors), np.array(gaps)
    out = {"pairs_considered": len(pairs), "pairs_measured": int(len(errors))}
    if len(errors):
        out.update(median_px=round(float(np.median(errors)), 2), p90_px=round(float(np.percentile(errors, 90)), 2),
                   max_px=round(float(errors.max()), 2))
        for lo, hi in ((1, 5), (5, 20), (20, 10 ** 6)):
            sel = (gaps >= lo) & (gaps < hi)
            if sel.any():
                out[f"median_px_gap_{lo}_{hi if hi < 10 ** 6 else 'plus'}"] = round(float(np.median(errors[sel])), 2)
    return out

In [ ]:
%%writefile /kaggle/working/gs_pipeline/postprocess.py
"""CPU post-processing of a trained splat for viewing: upright + centered, cropped to the object, faint
Gaussians dropped. Runs on the exported PLY, so no GPU re-run is needed.

Rotating a splat must also rotate its view-dependent color (SH bands 1-3); otherwise colors
"swim" when orbiting. Each band is rotated by a matrix solved numerically from the SH basis.
"""

from pathlib import Path

import numpy as np

from .manifest import load_frames
from .ply_io import read_gaussians_ply, write_gaussians_ply
from .validate import object_bounds, scene_target

# Real SH basis constants, same convention as 3DGS / gsplat.
C1 = 0.4886025119029199
C2 = [1.0925484305920792, -1.0925484305920792, 0.31539156525252005, -1.0925484305920792, 0.5462742152960396]
C3 = [-0.5900435899266435, 2.890611442640554, -0.4570457994644658, 0.3731763325901154,
      -0.4570457994644658, 1.445305721320277, -0.5900435899266435]
BANDS = [(0, 3), (3, 8), (8, 15)]  # index ranges of SH bands 1..3 inside shN


def sh_basis(d: np.ndarray) -> np.ndarray:
    """SH basis values (without the DC term) for unit directions d (M,3) -> (M,15)."""
    x, y, z = d[:, 0], d[:, 1], d[:, 2]
    xx, yy, zz = x * x, y * y, z * z
    return np.stack([
        -C1 * y, C1 * z, -C1 * x,
        C2[0] * x * y, C2[1] * y * z, C2[2] * (2 * zz - xx - yy), C2[3] * x * z, C2[4] * (xx - yy),
        C3[0] * y * (3 * xx - yy), C3[1] * x * y * z, C3[2] * y * (4 * zz - xx - yy),
        C3[3] * z * (2 * zz - 3 * xx - 3 * yy), C3[4] * x * (4 * zz - xx - yy), C3[5] * z * (xx - yy),
        C3[6] * x * (xx - 3 * yy),
    ], axis=1)


def sh_rotation_matrices(R: np.ndarray) -> list[np.ndarray]:
    """Per-band T such that coefficients c' = T @ c reproduce color(R d) == old color(d)."""
    d = np.random.default_rng(0).normal(size=(256, 3))
    d /= np.linalg.norm(d, axis=1, keepdims=True)
    A, B = sh_basis(d @ R.T), sh_basis(d)
    return [np.linalg.lstsq(A[:, a:b], B[:, a:b], rcond=None)[0] for a, b in BANDS]


def rotate_shN(shN: np.ndarray, R: np.ndarray) -> np.ndarray:
    out = shN.copy()
    for (a, b), T in zip(BANDS, sh_rotation_matrices(R)):
        if shN.shape[1] >= b:
            out[:, a:b] = np.einsum("ij,njc->nic", T, shN[:, a:b])
    return out


def rotmat_to_quat_wxyz(R: np.ndarray) -> np.ndarray:
    w = np.sqrt(max(0.0, 1 + R[0, 0] + R[1, 1] + R[2, 2])) / 2
    x = np.copysign(np.sqrt(max(0.0, 1 + R[0, 0] - R[1, 1] - R[2, 2])) / 2, R[2, 1] - R[1, 2])
    y = np.copysign(np.sqrt(max(0.0, 1 - R[0, 0] + R[1, 1] - R[2, 2])) / 2, R[0, 2] - R[2, 0])
    z = np.copysign(np.sqrt(max(0.0, 1 - R[0, 0] - R[1, 1] + R[2, 2])) / 2, R[1, 0] - R[0, 1])
    return np.array([w, x, y, z])


def quat_multiply(q1: np.ndarray, q2: np.ndarray) -> np.ndarray:
    """Hamilton product q1 * q2, wxyz; q2 may be (N,4)."""
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]
    return np.stack([w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2, w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
                     w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2, w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2], axis=-1)


def upright_rotation(up: np.ndarray) -> np.ndarray:
    """Smallest rotation taking `up` to +y."""
    a = up / np.linalg.norm(up)
    b = np.array([0.0, 1.0, 0.0])
    v, c = np.cross(a, b), float(a @ b)
    if np.linalg.norm(v) < 1e-9:
        return np.eye(3) if c > 0 else np.diag([1.0, -1.0, -1.0])
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * (1 / (1 + c))


def transform_gaussians(g: dict, R: np.ndarray, t: np.ndarray) -> dict:
    """Apply x -> R x + t to a splat (positions, orientations, view-dependent color)."""
    q = quat_multiply(rotmat_to_quat_wxyz(R), g["quats_wxyz"] / np.linalg.norm(g["quats_wxyz"], axis=1,
                                                                               keepdims=True))
    return {**g, "means": g["means"] @ R.T + t, "quats_wxyz": q, "shN": rotate_shN(g["shN"], R)}


def floater_mask(g: dict, k: int = 8, isolation: float = 4.0, faint: float = 0.35) -> np.ndarray:
    """True for Gaussians to keep. Floaters / background haze are isolated (mean distance to their k
    nearest neighbours > `isolation` x the median spacing) AND faint (opacity < `faint`). On the first
    steel-bottle capture they had 4% opacity at ~16 mm spacing vs. 20-30% at 1-3 mm on real surfaces."""
    from scipy.spatial import cKDTree

    means = g["means"]
    if len(means) <= k:
        return np.ones(len(means), bool)
    d, _ = cKDTree(means).query(means, k=k + 1)
    spacing = d[:, 1:].mean(axis=1)
    opacity = 1 / (1 + np.exp(-g["opacities_logit"]))
    return ~((spacing > isolation * np.median(spacing)) & (opacity < faint))


def needle_mask(g: dict, max_len: float = 0.25, max_ratio: float = 15.0) -> np.ndarray:
    """True for Gaussians to keep. Needles ("fur" on the first room's ceiling) are long (largest axis
    > max_len m) AND thin in both other directions (largest / middle axis > max_ratio). Real surfaces are
    flat discs (two long axes), so big discs on blank walls are kept."""
    s = np.sort(np.exp(g["scales_log"]), axis=1)
    return ~((s[:, 2] > max_len) & (s[:, 2] > max_ratio * np.maximum(s[:, 1], 1e-9)))


def clean(splat_ply, out_ply, **kw) -> dict:
    """Remove floaters from an existing splat file (no manifest needed)."""
    g = read_gaussians_ply(splat_ply)
    keep = floater_mask(g, **kw)
    write_gaussians_ply(out_ply, **{k: v[keep] for k, v in g.items()})
    return {"in": len(keep), "out": int(keep.sum()), "removed": int((~keep).sum()),
            "out_mb": round(Path(out_ply).stat().st_size / 1e6, 1)}


def postprocess(splat_ply, manifest_path, out_ply, radius: float | None = None, min_opacity: float = 0.02,
                remove_floaters: bool = True) -> dict:
    """Upright + center on the capture target, crop to a sphere, drop faint Gaussians.

    radius defaults to 1.3 x the bounding sphere of the (point-refined) object cylinder if present, else
    0.35 x median camera distance.
    """
    frames, manifest = load_frames(manifest_path)
    c2ws = [f.c2w_gl for f in frames]
    from .scene import capture_mode

    if capture_mode(manifest) == "scene":
        return postprocess_scene(splat_ply, manifest_path, manifest, c2ws, out_ply, min_opacity, remove_floaters)
    target, _ = scene_target(manifest, manifest_path, c2ws)
    up = np.mean([c[:3, 1] for c in c2ws], axis=0)  # camera y (OpenGL) = image up
    if radius is None:
        b = object_bounds(manifest_path, manifest)
        if b is not None:
            radius = 1.3 * b.sphere_radius
        else:
            radius = 0.35 * float(np.median([np.linalg.norm(c[:3, 3] - target) for c in c2ws]))

    g = read_gaussians_ply(splat_ply)
    n_in = len(g["means"])
    keep = (np.linalg.norm(g["means"] - target, axis=1) < radius)
    keep &= 1 / (1 + np.exp(-g["opacities_logit"])) >= min_opacity
    g = {k: v[keep] for k, v in g.items()}
    floaters = 0
    if remove_floaters:
        fk = floater_mask(g)
        floaters = int((~fk).sum())
        g = {k: v[fk] for k, v in g.items()}

    R = upright_rotation(up)
    g = transform_gaussians(g, R, -R @ target)
    write_gaussians_ply(out_ply, **g)
    return {"in": n_in, "out": len(g["means"]), "floaters_removed": floaters, "radius": round(radius, 4),
            "out_mb": round(Path(out_ply).stat().st_size / 1e6, 1)}


def postprocess_scene(splat_ply, manifest_path, manifest: dict, c2ws: list, out_ply, min_opacity: float = 0.02,
                      remove_floaters: bool = True) -> dict:
    """Rooms: keep everything inside the room box, put the floor at y = 0 and the room centre at x = z = 0.

    The AR world is already gravity-aligned (refine aligns COLMAP back to it), so no rotation is needed.
    Also returns a suggested first-person start view for the viewer (in output coordinates).
    The output file keeps the legacy name splat_object.ply so artifacts/app/viewer don't change.
    """
    from .scene import crop_box, floor_height, forward_gl
    from .validate import load_points

    centers = np.array([c[:3, 3] for c in c2ws])
    pts = load_points(manifest_path, manifest)
    lo, hi = crop_box(pts, centers)
    floor = floor_height(manifest, pts, centers)
    g = read_gaussians_ply(splat_ply)
    n_in = len(g["means"])
    keep = np.all((g["means"] >= lo) & (g["means"] <= hi), axis=1)
    keep &= 1 / (1 + np.exp(-g["opacities_logit"])) >= min_opacity
    nk = needle_mask(g)
    needles = int((keep & ~nk).sum())
    keep &= nk
    g = {k: v[keep] for k, v in g.items()}
    floaters = 0
    if remove_floaters:
        fk = floater_mask(g)
        floaters = int((~fk).sum())
        g = {k: v[fk] for k, v in g.items()}

    shift = -np.array([0.5 * (lo[0] + hi[0]), floor, 0.5 * (lo[2] + hi[2])])
    g["means"] = g["means"] + shift
    write_gaussians_ply(out_ply, **g)
    eye = np.median(centers, axis=0) + shift
    return {"in": n_in, "out": len(g["means"]), "floaters_removed": floaters, "needles_removed": needles,
            "box_m": [round(float(v), 2) for v in (hi - lo)],
            "view": {"eye": [round(float(v), 3) for v in eye],
                     "forward": [round(float(v), 4) for v in forward_gl(c2ws[0])]},
            "out_mb": round(Path(out_ply).stat().st_size / 1e6, 1)}

In [ ]:
%%writefile /kaggle/working/gs_pipeline/refine.py
"""Pose refinement: COLMAP re-estimates poses + lens from the photos; AR supplies scale, gravity and pairs.

Why: on the first real captures ARCore poses were off by 4-13 cm and its focal length by 11%, giving
10-37 px epipolar error, while COLMAP on the same photos reaches < 1 px. AR is still used for
(1) choosing which photos to match, (2) the starting focal length, (3) metric scale + gravity-up via a
similarity fit, and (4) the fallback when COLMAP cannot register enough photos.

    python -m gs_pipeline refine <bundle>/manifest.json --out <refined_dir>
"""

import json
import shutil
import time
from pathlib import Path

import numpy as np

from .manifest import load_frames, read_manifest
from .scene import capture_mode, select_pairs_scene
from .validate import scene_target

MIN_REGISTERED_FRACTION = 0.8
MAX_REPROJ_PX = 1.5
CARRY_OVER = ("capture_id", "device", "object_bounds", "capture_mode")
FRAME_CARRY_OVER = ("target_id", "blur_score", "timestamp")  # per photo: which spot/sector, how sharp


# ---------------------------------------------------------------- pure helpers (unit-tested)

def select_pairs(names: list[str], centers: np.ndarray, target: np.ndarray, k_nearest: int = 12,
                 time_window: int = 2) -> list[tuple[str, str]]:
    """Image pairs worth matching: nearest viewing directions around the target + neighbours in time."""
    d = centers - target
    d /= np.linalg.norm(d, axis=1, keepdims=True)
    sim = d @ d.T
    pairs = set()
    n = len(names)
    for i in range(n):
        for j in np.argsort(-sim[i])[1:k_nearest + 1]:
            pairs.add((min(i, int(j)), max(i, int(j))))
        for j in range(i + 1, min(n, i + time_window + 1)):
            pairs.add((i, j))
    return [(names[i], names[j]) for i, j in sorted(pairs)]


def umeyama(src: np.ndarray, dst: np.ndarray) -> tuple[float, np.ndarray, np.ndarray]:
    """Similarity (s, R, t) minimising |s R src + t - dst|^2."""
    mu_s, mu_d = src.mean(0), dst.mean(0)
    a, b = src - mu_s, dst - mu_d
    U, D, Vt = np.linalg.svd(b.T @ a / len(src))
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt
    s = float(np.trace(np.diag(D) @ S) / a.var(0).sum())
    return s, R, mu_d - s * R @ mu_s


def robust_similarity(src: np.ndarray, dst: np.ndarray, trim: float = 0.2):
    """Umeyama, then refit without the worst `trim` fraction. Returns (s, R, t, residuals_of_all)."""
    s, R, t = umeyama(src, dst)
    res = np.linalg.norm((s * (R @ src.T)).T + t - dst, axis=1)
    keep = res <= np.quantile(res, 1 - trim)
    if keep.sum() >= 3:
        s, R, t = umeyama(src[keep], dst[keep])
        res = np.linalg.norm((s * (R @ src.T)).T + t - dst, axis=1)
    return s, R, t, res


def carry_frame_fields(refined: dict, manifest: dict) -> None:
    """Copy per-photo app fields (spot/sector target_id, blur score, time) onto the refined frames, by file name."""
    orig = {Path(f["file_path"]).name: f for f in manifest.get("frames", [])}
    for f in refined.get("frames", []):
        o = orig.get(Path(f["file_path"]).name)
        if o:
            for k in FRAME_CARRY_OVER:
                if k in o:
                    f[k] = o[k]


# ---------------------------------------------------------------- pipeline

def _camera_center(image) -> np.ndarray:
    m = np.eye(4)
    m[:3, :4] = image.cam_from_world().matrix()
    return np.linalg.inv(m)[:3, 3]


def refine(manifest_path, out_dir, num_threads: int = 2, max_image_size: int = 1600,
           k_nearest: int = 12) -> dict:
    """Write a refined bundle to out_dir. Returns the report; report["manifest"] is the manifest to
    train on (the refined one, or the original AR manifest if refinement fell back)."""
    import pycolmap

    manifest_path = Path(manifest_path).resolve()
    bundle = manifest_path.parent
    out_dir = Path(out_dir).resolve()
    shutil.rmtree(out_dir, ignore_errors=True)
    work = out_dir / "_colmap"
    work.mkdir(parents=True)
    frames, manifest = load_frames(manifest_path)
    report = {"total_images": len(frames), "fallback": False}
    t0 = time.time()

    # Images must share one folder for COLMAP; names are relative to it.
    image_dir = frames[0].image_path.parent
    if any(f.image_path.parent != image_dir for f in frames):
        raise ValueError("refine expects all images in one folder")
    names = [f.image_path.name for f in frames]
    scene = capture_mode(manifest) == "scene"
    report["capture_mode"] = capture_mode(manifest)

    try:
        db = work / "database.db"
        reader = pycolmap.ImageReaderOptions()
        reader.camera_model = "SIMPLE_RADIAL"
        f0 = float(np.median([f.K[0, 0] for f in frames]))
        w, h = frames[0].width, frames[0].height
        reader.camera_params = f"{f0},{w / 2},{h / 2},0"  # AR focal as the starting guess
        ext = pycolmap.FeatureExtractionOptions()
        ext.num_threads, ext.max_image_size, ext.use_gpu = num_threads, max_image_size, False
        ext.sift.max_num_features = 8000 if scene else 6000
        pycolmap.extract_features(db, image_dir, image_names=names, camera_mode=pycolmap.CameraMode.SINGLE,
                                  reader_options=reader, extraction_options=ext, device=pycolmap.Device.cpu)
        report["t_features_s"] = round(time.time() - t0, 1)

        t1 = time.time()
        if scene:  # no single target: pair cameras that look at the same part of the room
            pairs = select_pairs_scene(names, [f.c2w_gl for f in frames])
        else:
            centers = np.array([f.c2w_gl[:3, 3] for f in frames])
            target, _ = scene_target(manifest, manifest_path, [f.c2w_gl for f in frames])
            pairs = select_pairs(names, centers, target, k_nearest=k_nearest)
        (work / "pairs.txt").write_text("\n".join(f"{a} {b}" for a, b in pairs))
        mo = pycolmap.FeatureMatchingOptions()
        mo.num_threads, mo.use_gpu = num_threads * 2, False
        po = pycolmap.ImportedPairingOptions()
        po.match_list_path = str(work / "pairs.txt")
        pycolmap.match_image_pairs(db, matching_options=mo, pairing_options=po, device=pycolmap.Device.cpu)
        report["pairs"] = len(pairs)
        report["t_matching_s"] = round(time.time() - t1, 1)

        t2 = time.time()
        recs = pycolmap.incremental_mapping(db, image_dir, work / "sparse")
        report["t_mapping_s"] = round(time.time() - t2, 1)
        if not recs:
            raise RuntimeError("COLMAP produced no model")
        rec = max(recs.values(), key=lambda r: r.num_reg_images())

        reg = rec.num_reg_images()
        reproj = float(rec.compute_mean_reprojection_error())
        cam = next(iter(rec.cameras.values()))
        report.update(registered=reg, reproj_px=round(reproj, 3), points=rec.num_points3D(),
                      focal_ar_px=round(f0, 1), focal_colmap_px=round(float(cam.params[0]), 1),
                      radial_k=round(float(cam.params[3]), 5))
        if reg < MIN_REGISTERED_FRACTION * len(frames) or reproj > MAX_REPROJ_PX:
            raise RuntimeError(f"quality gate failed: {reg}/{len(frames)} registered, {reproj:.2f} px")

        # COLMAP world -> AR world (metres, gravity-up), fitted on camera centres.
        by_name = {f.image_path.name: f for f in frames}
        src = np.array([_camera_center(im) for im in rec.images.values()])
        dst = np.array([by_name[im.name].c2w_gl[:3, 3] for im in rec.images.values()])
        s, R, t, res = robust_similarity(src, dst)
        rec.transform(pycolmap.Sim3d(np.c_[s * R, t]))
        report.update(ar_drift_cm={"median": round(100 * float(np.median(res)), 1),
                                   "p90": round(100 * float(np.percentile(res, 90)), 1),
                                   "max": round(100 * float(res.max()), 1)},
                      scale_colmap_to_m=round(s, 5))
        aligned = work / "aligned"
        aligned.mkdir()
        rec.write_binary(str(aligned))

        # Undistort to PINHOLE (the trainer's model), then reuse the COLMAP -> manifest converter.
        t3 = time.time()
        pycolmap.undistort_images(out_dir / "undistorted", aligned, image_dir)
        from .colmap_io import convert_colmap

        und = out_dir / "undistorted"
        convert_colmap(und, images_subdir="images", sparse_subdir="sparse")
        shutil.move(str(und / "images"), str(out_dir / "images"))
        shutil.move(str(und / "points3d.ply"), str(out_dir / "points3d.ply"))
        refined = read_manifest(und / "manifest.json")
        report["t_undistort_s"] = round(time.time() - t3, 1)
    except Exception as exc:  # noqa: BLE001 - any failure means: train on the AR poses instead
        report.update(fallback=True, error=f"{type(exc).__name__}: {exc}")
        report["manifest"] = str(manifest_path)
        report["t_total_s"] = round(time.time() - t0, 1)
        (out_dir / "refine_report.json").write_text(json.dumps(report, indent=2))
        return report

    for key in CARRY_OVER:
        if key in manifest:
            refined[key] = manifest[key]
    carry_frame_fields(refined, manifest)
    refined["source"] = "arcore+colmap" if manifest.get("source") in ("arcore", "arkit") else "colmap"
    report["t_total_s"] = round(time.time() - t0, 1)
    refined.setdefault("extensions", {})
    refined["extensions"].update(manifest.get("extensions", {}))
    refined["extensions"]["refine"] = report
    report["manifest"] = str(out_dir / "manifest.json")
    (out_dir / "manifest.json").write_text(json.dumps(refined, indent=1))
    (out_dir / "refine_report.json").write_text(json.dumps(report, indent=2))
    shutil.rmtree(und, ignore_errors=True)
    shutil.rmtree(work, ignore_errors=True)
    return report

In [ ]:
%%writefile /kaggle/working/gs_pipeline/scene.py
"""Scene captures (whole rooms): the user stands at a few spots ("stations") and turns with the phone at
arm's length, so the lens moves on ~0.8 m circles. Object-only assumptions (one look-at target, an
orbit, a crop sphere) don't apply; these helpers replace them.

Depth needs the lens to move: photos taken while only rotating about the lens are a panorama, so COLMAP
can't triangulate and 3DGS turns into a blurry bubble. `parallax_check` rejects that early.
"""

import numpy as np

LOOK_DEPTH_M = 1.5          # typical distance to what a room photo shows (walls 1-4 m away)
MIN_SPREAD_M = 0.15         # all lenses within this of their median = pure rotation
LOW_SPREAD_M = 0.4


def capture_mode(manifest: dict) -> str:
    """'scene' or 'object' (captures from before schema 1.1 are objects)."""
    return "scene" if manifest.get("capture_mode") == "scene" else "object"


def forward_gl(c2w: np.ndarray) -> np.ndarray:
    """Viewing direction of an OpenGL camera-to-world (camera looks down -z)."""
    f = -c2w[:3, 2]
    return f / np.linalg.norm(f)


def select_pairs_scene(names: list[str], c2ws: list[np.ndarray], k_nearest: int = 12, k_cross: int = 12,
                       time_window: int = 4, max_angle_deg: float = 70.0, cross_min_m: float = 0.6
                       ) -> list[tuple[str, str]]:
    """Image pairs worth matching in a room: neighbours in time + cameras that look at the same spot.

    "Same spot" = distance between points LOOK_DEPTH_M in front of each camera, among cameras whose
    viewing directions are within max_angle_deg. That covers both views from one station and views of
    the same wall from different stations (incl. loop closure when the user returns to the start).
    The nearest look-points are almost always from the same station, so k_cross extra pairs are taken
    only among cameras >= cross_min_m away; without them stations would be reconstructed separately.
    """
    n = len(names)
    centers = np.array([c[:3, 3] for c in c2ws])
    fwd = np.array([forward_gl(c) for c in c2ws])
    look = centers + LOOK_DEPTH_M * fwd
    dist = np.linalg.norm(look[:, None] - look[None], axis=-1)
    dist[fwd @ fwd.T < np.cos(np.radians(max_angle_deg))] = np.inf
    far = np.linalg.norm(centers[:, None] - centers[None], axis=-1) >= cross_min_m
    pairs = set()
    for i in range(n):
        near = [j for j in np.argsort(dist[i])[:k_nearest + 1] if j != i]
        cross = [j for j in np.argsort(np.where(far[i], dist[i], np.inf))[:k_cross]]
        for j in near + cross:
            if np.isfinite(dist[i, j]):
                pairs.add((min(i, int(j)), max(i, int(j))))
        for j in range(i + 1, min(n, i + time_window + 1)):
            pairs.add((i, j))
    return [(names[i], names[j]) for i, j in sorted(pairs)]


def camera_spread(centers: np.ndarray) -> float:
    """How far the lens travelled: 90th percentile distance of camera centres from their median (m)."""
    return float(np.percentile(np.linalg.norm(centers - np.median(centers, axis=0), axis=1), 90))


def parallax_check(centers: np.ndarray) -> tuple[list[str], list[str], float]:
    """(errors, warnings, spread)."""
    spread = camera_spread(centers)
    errors, warnings = [], []
    if spread < MIN_SPREAD_M:
        errors.append(f"cameras barely moved (spread {spread * 100:.0f} cm): turning in place gives no depth. "
                      "Hold the phone at arm's length while turning, and capture from several spots")
    elif spread < LOW_SPREAD_M:
        warnings.append(f"cameras moved only {spread * 100:.0f} cm; capture from more spots for better depth")
    return errors, warnings, spread


def scene_info(manifest: dict) -> dict:
    return (manifest.get("extensions") or {}).get("scene") or {}


def crop_box(points: np.ndarray | None, centers: np.ndarray, lo: float = 2.0, hi: float = 98.0,
             margin: float = 0.5) -> tuple[np.ndarray, np.ndarray]:
    """Axis-aligned box holding the room: robust extent of the sparse points (+ margin) and all cameras."""
    if points is not None and len(points) >= 50:
        a, b = np.percentile(points, lo, axis=0) - margin, np.percentile(points, hi, axis=0) + margin
    else:
        a, b = centers.min(0) - 3.0, centers.max(0) + 3.0
    return np.minimum(a, centers.min(0) - margin), np.maximum(b, centers.max(0) + margin)


def floor_height(manifest: dict, points: np.ndarray | None, centers: np.ndarray) -> float:
    """Floor y: the app's AR floor plane if recorded, else the 2nd percentile of points, else 1.4 m below
    the cameras (phone held at chest height)."""
    y = scene_info(manifest).get("floor_y")
    if y is not None:
        return float(y)
    if points is not None and len(points) >= 50:
        return float(np.percentile(points[:, 1], 2))
    return float(np.median(centers[:, 1]) - 1.4)

In [ ]:
%%writefile /kaggle/working/gs_pipeline/train.py
"""3D Gaussian Splatting trainer on gsplat. Manifest in -> splat.ply + metrics.json + previews out.

Follows gsplat's simple_trainer defaults (learning rates, DefaultStrategy or MCMCStrategy densification,
L1 + SSIM loss, SH degree ramp-up). Rooms (the scene profile) add a monocular depth loss, per-photo
exposure compensation, camera pose refinement and a min-views prune; see train_config.PROFILES.
`--dry-run` swaps the CUDA rasterizer for a tiny torch stand-in so the whole loop (data, init,
densification wiring, export) can be checked on a CPU before spending GPU time.
"""

import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from .cameras import scale_intrinsics, viewmat_from_c2w_gl
from .manifest import load_frames, points_path
from .ply_io import read_points_ply, write_gaussians_ply

SH_C0 = 0.28209479177387814


def rgb_to_sh(rgb: torch.Tensor) -> torch.Tensor:
    return (rgb - 0.5) / SH_C0


def load_images(frames, max_side: int):
    """uint8 HxWx3 tensors + intrinsics rescaled to the loaded size."""
    from PIL import Image

    images, Ks = [], []
    for fr in frames:
        with Image.open(fr.image_path) as im:
            im = im.convert("RGB")
            if im.size != (fr.width, fr.height):
                raise ValueError(f"{fr.image_path.name}: image is {im.size}, manifest says {(fr.width, fr.height)}")
            s = min(1.0, max_side / max(im.size))
            if s < 1.0:
                im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
            Ks.append(scale_intrinsics(fr.K, im.width / fr.width, im.height / fr.height))
            images.append(torch.from_numpy(np.asarray(im).copy()))
    return images, Ks


def init_points(manifest_path, manifest, centers, scene_scale, args, rng, dense_ply=None):
    """Initial Gaussian centers + colors: points3d.ply (+ depth-prior points from dense_ply) if present,
    else random in the object bounds."""
    pp = points_path(manifest_path, manifest)
    if pp is not None and pp.exists():
        xyz, rgb = read_points_ply(pp)
        if rgb is None:
            rgb = np.full((len(xyz), 3), 128, np.uint8)
        src = f"points3d.ply ({len(xyz)} pts)"
        if dense_ply is not None and Path(dense_ply).exists():
            dxyz, drgb = read_points_ply(dense_ply)
            drgb = drgb if drgb is not None else np.full((len(dxyz), 3), 128, np.uint8)
            xyz, rgb = np.concatenate([xyz, dxyz]), np.concatenate([rgb, drgb])
            src += f" + depth prior ({len(dxyz)} pts)"
        if len(xyz) > args.max_init_points:
            keep = rng.choice(len(xyz), args.max_init_points, replace=False)
            xyz, rgb = xyz[keep], rgb[keep]
            src += f", subsampled to {len(xyz)}"
        return xyz.astype(np.float32), rgb.astype(np.float32) / 255.0, src
    if "object_bounds" in manifest:
        b = manifest["object_bounds"]
        center = np.asarray(b["center"], float) + [0.0, 0.5 * b.get("height", 0.0), 0.0]
        extent = 1.5 * float(np.hypot(b["radius"], 0.5 * b.get("height", 0.0)))
    else:
        center, extent = centers.mean(0), scene_scale
    xyz = center + (rng.random((args.init_random, 3)) * 2 - 1) * extent
    return xyz.astype(np.float32), rng.random((args.init_random, 3)).astype(np.float32), f"random ({args.init_random} pts)"


def knn_scale_log(xyz: np.ndarray) -> np.ndarray:
    """log of RMS distance to the 3 nearest neighbours, as the initial isotropic scale."""
    from scipy.spatial import cKDTree

    d, _ = cKDTree(xyz).query(xyz, k=4)
    dist = np.sqrt((d[:, 1:] ** 2).mean(1)).clip(1e-7)
    return np.repeat(np.log(dist)[:, None], 3, axis=1).astype(np.float32)


def _gaussian_kernel(size=11, sigma=1.5, device="cpu"):
    x = torch.arange(size, device=device, dtype=torch.float32) - size // 2
    g = torch.exp(-x ** 2 / (2 * sigma ** 2))
    g = g / g.sum()
    return (g[:, None] * g[None, :]).expand(3, 1, size, size).contiguous()


def ssim(a: torch.Tensor, b: torch.Tensor, kernel: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
    """Mean SSIM of two (1,3,H,W) images in [0,1], optionally only over mask (H,W)."""
    pad = kernel.shape[-1] // 2
    conv = lambda x: F.conv2d(x, kernel, padding=pad, groups=3)  # noqa: E731
    mu_a, mu_b = conv(a), conv(b)
    s_aa, s_bb, s_ab = conv(a * a) - mu_a ** 2, conv(b * b) - mu_b ** 2, conv(a * b) - mu_a * mu_b
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    smap = ((2 * mu_a * mu_b + c1) * (2 * s_ab + c2)) / ((mu_a ** 2 + mu_b ** 2 + c1) * (s_aa + s_bb + c2))
    if mask is None:
        return smap.mean()
    return (smap * mask).sum() / (mask.sum() * 3).clamp(min=1)


def masked_mean(x: torch.Tensor, mask: torch.Tensor | None) -> torch.Tensor:
    """Mean of an (H,W,3) tensor over mask (H,W); plain mean without a mask."""
    if mask is None:
        return x.mean()
    return (x * mask[..., None]).sum() / (mask.sum() * 3).clamp(min=1)


def region_masks(frames, Ks, sizes, center: np.ndarray, radius: float, device) -> list[torch.Tensor]:
    """Per image: pixels covered by the region sphere's silhouette (slightly enlarged, conservative)."""
    masks = []
    for fr, K, (h, w) in zip(frames, Ks, sizes):
        vm = viewmat_from_c2w_gl(fr.c2w_gl)
        pc = vm[:3, :3] @ center + vm[:3, 3]
        d = float(np.linalg.norm(pc))
        if d <= radius * 1.05 or pc[2] <= 0:
            masks.append(torch.ones(h, w, device=device))
            continue
        u, v = K[0, 0] * pc[0] / pc[2] + K[0, 2], K[1, 1] * pc[1] / pc[2] + K[1, 2]
        r_px = 1.15 * max(K[0, 0], K[1, 1]) * np.tan(np.arcsin(min(radius / d, 0.999))) * d / pc[2]
        yy, xx = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing="ij")
        masks.append((((xx + 0.5 - u) ** 2 + (yy + 0.5 - v) ** 2) <= r_px ** 2).float())
    return masks


def fake_rasterization(means, quats, scales, opacities, colors, viewmats, Ks, width, height, sh_degree=None,
                       packed=False, render_mode="RGB", **_):
    """CPU stand-in for gsplat.rasterization with matching output shapes (dry runs only)."""
    C, N = viewmats.shape[0], means.shape[0]
    cam = torch.einsum("cij,nj->cni", viewmats[:, :3, :3], means) + viewmats[:, None, :3, 3]
    z = cam[..., 2:].clamp(min=1e-3)
    means2d = cam[..., :2] / z * torch.stack([Ks[:, 0, 0], Ks[:, 1, 1]], -1)[:, None] + Ks[:, None, :2, 2]
    radii = torch.where(cam[..., 2:] > 0, 3, 0).expand(C, N, 2).int()
    color = (colors[:, 0] * SH_C0 + 0.5).mean(0) * opacities.mean() * scales.mean() * quats.norm(dim=-1).mean()
    render = color.expand(C, height, width, 3) + 1e-6 * means2d.tanh().sum()  # bounded: z ~ 0 projects to ~1e5 px
    if render_mode == "RGB+ED":
        far = cam[..., 2].amax(1).clamp(min=0.1)  # a wall behind everything: nothing is occluded
        render = torch.cat([render, far[:, None, None, None].expand(C, height, width, 1)], -1)
    info = {"means2d": means2d, "radii": radii, "width": width, "height": height, "n_cameras": C,
            "gaussian_ids": None}
    return render, torch.ones(C, height, width, 1), info


def focus_strategy_class(base):
    """DefaultStrategy that only clones/splits Gaussians inside the object region, and/or stops
    growing once `cap_max` Gaussians exist (rooms on a 16 GB T4).

    Background Gaussians are kept (they must explain the room seen behind the object; removing them
    made the trainer fake the background with floaters around the object) but stay coarse.
    """
    from dataclasses import dataclass

    @dataclass
    class FocusStrategy(base):
        region_center: torch.Tensor | None = None
        region_radius: float = 0.0
        cap_max: int = 0

        def _grow_gs(self, params, optimizers, state, step):
            if self.cap_max and len(params["means"]) >= self.cap_max:
                return 0, 0  # (n_dupli, n_split): pruning and opacity resets still run
            if self.region_center is not None:
                with torch.no_grad():
                    outside = (params["means"] - self.region_center).norm(dim=-1) > self.region_radius
                    state["grad2d"][outside] = 0.0
            return super()._grow_gs(params, optimizers, state, step)

    return FocusStrategy


def dry_mcmc_class(base):
    """MCMCStrategy whose relocation / noise steps (CUDA kernels) are skipped, for CPU dry runs."""

    class DryMCMC(base):
        def step_post_backward(self, params, optimizers, state, step, info, lr):
            pass

    return DryMCMC


def se3_exp(delta: torch.Tensor) -> torch.Tensor:
    """(6,) axis-angle + translation -> 4x4 rigid transform (differentiable, safe at zero)."""
    w, t = delta[:3], delta[3:]
    theta = torch.sqrt((w * w).sum() + 1e-12)
    zero = torch.zeros((), dtype=delta.dtype, device=delta.device)
    Kx = torch.stack([torch.stack([zero, -w[2], w[1]]), torch.stack([w[2], zero, -w[0]]),
                      torch.stack([-w[1], w[0], zero])])
    R = torch.eye(3, dtype=delta.dtype, device=delta.device) + torch.sin(theta) / theta * Kx \
        + (1 - torch.cos(theta)) / theta ** 2 * (Kx @ Kx)
    bottom = torch.tensor([[0.0, 0.0, 0.0, 1.0]], dtype=delta.dtype, device=delta.device)
    return torch.cat([torch.cat([R, t[:, None]], 1), bottom], 0)


def depth_loss_weight(step: int, steps: int, start: float, final: float) -> float:
    """Exponential decay start -> final: strong early (shape), weak late (photo detail wins)."""
    if start <= 0:
        return 0.0
    return float(start * (max(final, 1e-12) / start) ** (step / max(steps - 1, 1)))


def visible_in_view(means: torch.Tensor, viewmat: torch.Tensor, K: torch.Tensor, radii: torch.Tensor,
                    depth: torch.Tensor, slack: float = 0.05) -> torch.Tensor:
    """Bool (N,): Gaussians drawn in this view (radii > 0) whose centre is not hidden behind the rendered
    surface (centre depth <= rendered depth x (1 + slack))."""
    h, w = depth.shape
    cam = means @ viewmat[:3, :3].T + viewmat[:3, 3]
    z = cam[:, 2]
    u = K[0, 0] * cam[:, 0] / z.clamp(min=1e-6) + K[0, 2]
    v = K[1, 1] * cam[:, 1] / z.clamp(min=1e-6) + K[1, 2]
    drawn = (radii.reshape(len(means), -1) > 0).any(-1)
    inside = drawn & (z > 0) & (u >= 0) & (u < w) & (v >= 0) & (v < h)
    ui, vi = u.clamp(0, w - 1).long(), v.clamp(0, h - 1).long()
    return inside & (z <= depth[vi, ui] * (1 + slack))


def color_matched_psnr(img: torch.Tensor, gt: torch.Tensor) -> float:
    """PSNR after the best affine colour map (3x3 + bias, least squares) from render to photo. With per-photo
    exposure compensation the held-out photos' own exposure is unknown, so this separates "wrong colours"
    (exposure / white balance) from "wrong content" (geometry, holes, blur)."""
    x = torch.cat([img.reshape(-1, 3), torch.ones_like(img.reshape(-1, 3)[:, :1])], 1).double()
    A = torch.linalg.lstsq(x, gt.reshape(-1, 3).double()).solution
    fit = (x @ A).float().reshape(img.shape).clamp(0, 1)
    return float(-10 * torch.log10(F.mse_loss(fit, gt)))


def train(args) -> dict:
    import gsplat
    from gsplat import DefaultStrategy, MCMCStrategy

    rasterize = fake_rasterization if args.dry_run else gsplat.rasterization
    device = "cpu" if args.dry_run else "cuda"
    if not args.dry_run and not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU required (use --dry-run for a CPU smoke test)")
    torch.manual_seed(args.seed)
    rng = np.random.default_rng(args.seed)
    t0 = time.time()

    out = Path(args.out)
    (out / "previews").mkdir(parents=True, exist_ok=True)
    frames, manifest = load_frames(args.manifest)
    from .train_config import resolve_profile

    profile = resolve_profile(args, manifest)
    images, Ks = load_images(frames, args.max_side)
    n = len(frames)
    test_ids = list(range(0, n, args.test_every)) if args.test_every > 0 and n >= 2 * args.test_every else []
    train_ids = [i for i in range(n) if i not in set(test_ids)]
    images = [im.to(device) for im in images]
    viewmats = torch.tensor(np.stack([viewmat_from_c2w_gl(f.c2w_gl) for f in frames]), dtype=torch.float32,
                            device=device)
    Ks_t = torch.tensor(np.stack(Ks), dtype=torch.float32, device=device)
    centers = np.stack([f.c2w_gl[:3, 3] for f in frames])
    scene_scale = float(np.linalg.norm(centers - centers.mean(0), axis=1).max() * 1.1)

    # Monocular depth priors (scenes): aligned inverse depth per training photo + dense init points.
    from . import depth as depth_mod

    depth_dir = Path(args.depth_dir) if args.depth_dir else Path(args.manifest).parent / "depth"
    use_depth = args.depth != "off" and depth_mod.has_depth(depth_dir)
    if args.depth != "off" and not use_depth:
        print(f"[train] WARNING: no depth priors in {depth_dir} (run `gs_pipeline depth` first); "
              "blank walls and ceilings may get holes", flush=True)
    priors = [None] * n
    if use_depth:
        train_set = set(train_ids)
        for i, d in enumerate(depth_mod.load_aligned(depth_dir, frames)):
            if d is not None and i in train_set:
                inv = np.where(d > 0, 1.0 / np.maximum(d, 1e-6), 0.0)
                priors[i] = torch.tensor(inv, dtype=torch.float32, device=device)
    n_priors = sum(p is not None for p in priors)

    mcmc = args.strategy == "mcmc"
    xyz, rgb, init_src = init_points(args.manifest, manifest, centers, scene_scale, args, rng,
                                     dense_ply=depth_dir / depth_mod.DENSE_FILE if use_depth else None)

    # Object-focused training: only a sphere around the (point-refined) object is reconstructed and
    # supervised. On capture #2 the whole-scene run spent ~94% of its Gaussians (and time) on the room.
    region = None
    if args.focus == "auto" and not mcmc:
        from .validate import object_bounds

        b = object_bounds(args.manifest, manifest)
        if b is not None:
            region = (b.middle.astype(np.float32), float(args.focus_scale * b.sphere_radius))
    masks = [None] * n  # object-region masks, used only to report object PSNR
    if region is not None:
        r_center, r_radius = region
        inside = int((np.linalg.norm(xyz - r_center, axis=1) < r_radius).sum())
        init_src += f", densify only inside r={r_radius:.3f} m ({inside} of {len(xyz)} init pts inside)"
        # Only the held-out views need masks (object PSNR); full-res masks for every frame cost ~8 MB each.
        for i, m in zip(test_ids, region_masks([frames[i] for i in test_ids], [Ks[i] for i in test_ids],
                                               [images[i].shape[:2] for i in test_ids], r_center, r_radius, device)):
            masks[i] = m
    N = len(xyz)
    deg = args.sh_degree
    # MCMC starts from smaller, more opaque Gaussians (gsplat simple_trainer's MCMC settings).
    scales0 = knn_scale_log(xyz) + (np.float32(np.log(0.1)) if mcmc else np.float32(0.0))
    params = torch.nn.ParameterDict({
        "means": torch.nn.Parameter(torch.from_numpy(xyz)),
        "scales": torch.nn.Parameter(torch.from_numpy(scales0)),
        "quats": torch.nn.Parameter(torch.rand(N, 4)),
        "opacities": torch.nn.Parameter(torch.logit(torch.full((N,), 0.5 if mcmc else 0.1))),
        "sh0": torch.nn.Parameter(rgb_to_sh(torch.from_numpy(rgb))[:, None, :]),
        "shN": torch.nn.Parameter(torch.zeros(N, (deg + 1) ** 2 - 1, 3)),
    }).to(device)
    lrs = {"means": 1.6e-4 * scene_scale, "scales": 5e-3, "quats": 1e-3, "opacities": 5e-2,
           "sh0": 2.5e-3, "shN": 2.5e-3 / 20}
    optimizers = {k: torch.optim.Adam([{"params": params[k], "lr": lr, "name": k}], eps=1e-15)
                  for k, lr in lrs.items()}
    means_sched = torch.optim.lr_scheduler.ExponentialLR(optimizers["means"], gamma=0.01 ** (1.0 / max(args.steps, 1)))

    # Per-photo colour transform (3x3 + bias, identity at start): phones re-expose as you turn, which
    # otherwise shows up as blotchy walls. Used only in the loss; held-out views and the PLY never see it.
    extra_opts = []
    color_M = color_b = pose_delta = None
    if args.appearance == "on":
        color_M = torch.nn.Parameter(torch.eye(3, device=device).repeat(n, 1, 1))
        color_b = torch.nn.Parameter(torch.zeros(n, 3, device=device))
        extra_opts.append(torch.optim.Adam([color_M, color_b], lr=1e-3))
    # Pose refinement: a small rigid correction per training photo (held-out photos stay fixed).
    if args.pose_opt == "on":
        pose_delta = torch.nn.Parameter(torch.zeros(n, 6, device=device))
        extra_opts.append(torch.optim.Adam([pose_delta], lr=1e-5, weight_decay=1e-6))

    refine_stop = min(25_000 if mcmc else 15_000, int(args.steps * 0.85))
    if mcmc:
        cls = dry_mcmc_class(MCMCStrategy) if args.dry_run else MCMCStrategy
        strategy = cls(cap_max=args.cap_max or 1_000_000, refine_stop_iter=refine_stop)
        state = strategy.initialize_state()
    else:
        if region is not None or args.cap_max:
            strategy = focus_strategy_class(DefaultStrategy)(
                refine_stop_iter=refine_stop, cap_max=args.cap_max,
                region_center=None if region is None else torch.tensor(region[0], device=device),
                region_radius=0.0 if region is None else region[1])
        else:
            strategy = DefaultStrategy(refine_stop_iter=refine_stop)
        state = strategy.initialize_state(scene_scale=scene_scale)
    strategy.check_sanity(params, optimizers)
    kernel = _gaussian_kernel(device=device)
    raster_mode = "antialiased" if args.antialiased == "on" else "classic"
    train_mode = "RGB+ED" if n_priors else "RGB"

    print(f"[train] {n} frames ({len(train_ids)} train / {len(test_ids)} test), "
          f"{images[0].shape[1]}x{images[0].shape[0]} px, init {init_src}, scene_scale {scene_scale:.3f}, "
          f"device {device}, gsplat {gsplat.__version__}, profile {profile} ({args.steps} steps, "
          f"{args.strategy}, depth priors {n_priors}/{len(train_ids)}, appearance {args.appearance}, "
          f"pose-opt {args.pose_opt}, {raster_mode})", flush=True)

    def view_of(i, train_pose=False):
        if train_pose and pose_delta is not None:
            return se3_exp(pose_delta[i]) @ viewmats[i]
        return viewmats[i]

    def raster(vm, K, w, h, sh_deg, mode="RGB"):
        colors = torch.cat([params["sh0"], params["shN"]], 1)
        img, alpha, info = rasterize(
            means=params["means"], quats=params["quats"], scales=torch.exp(params["scales"]),
            opacities=torch.sigmoid(params["opacities"]), colors=colors, viewmats=vm[None], Ks=K[None],
            width=w, height=h, sh_degree=sh_deg, packed=False, render_mode=mode, rasterize_mode=raster_mode)
        return img[0], alpha[0, ..., 0], info

    def render(i, sh_deg, mode="RGB", train_pose=False):
        h, w = images[i].shape[:2]
        return raster(view_of(i, train_pose), Ks_t[i], w, h, sh_deg, mode)

    train_t = torch.tensor(train_ids, device=device)
    # The min-views prune runs inside training (not after it): the first room pruned 13% of the Gaussians
    # after the last step and lost held-out PSNR to the gaps; pruned this early, the rest adapt.
    prune_step = int(args.steps * 0.8) if args.min_views > 0 else -1
    prune = None
    perm, t_train = [], time.time()
    for step in range(args.steps):
        if not perm:
            perm = list(rng.permutation(train_ids))
        i = int(perm.pop())
        gt = images[i].float() / 255.0
        out_img, _, info = render(i, min(step // 1000, deg), train_mode, train_pose=True)
        strategy.step_pre_backward(params, optimizers, state, step, info)
        img = out_img[..., :3]
        if color_M is not None:
            img = img @ color_M[i].T + color_b[i]
        l1 = (img - gt).abs().mean()
        s = ssim(img.permute(2, 0, 1)[None], gt.permute(2, 0, 1)[None], kernel)
        loss = (1 - args.ssim_lambda) * l1 + args.ssim_lambda * (1 - s)
        if priors[i] is not None:
            inv = 1.0 / out_img[..., 3].clamp(min=1e-2)
            inv = F.interpolate(inv[None, None], size=priors[i].shape, mode="area")[0, 0]
            valid = priors[i] > 0
            lam = depth_loss_weight(step, args.steps, args.depth_lambda, args.depth_lambda_final)
            loss = loss + lam * (inv - priors[i])[valid].abs().mean()
        if mcmc:
            loss = loss + args.opacity_reg * torch.sigmoid(params["opacities"]).mean() \
                + args.scale_reg * torch.exp(params["scales"]).mean()
        if color_M is not None:
            loss = loss + 1e-3 * ((color_M[i] - torch.eye(3, device=device)) ** 2).sum() + 1e-3 * (color_b[i] ** 2).sum()
        loss.backward()
        for opt in list(optimizers.values()) + extra_opts:
            opt.step()
            opt.zero_grad(set_to_none=True)
        if color_M is not None:
            # Anchor: the mean per-photo transform stays identity, so the splat's colours match the average
            # photo. Unanchored, colour drifted into the transforms and held-out views (rendered without one)
            # lost ~3 dB on the first room.
            with torch.no_grad():
                color_M -= color_M[train_t].mean(0) - torch.eye(3, device=device)
                color_b -= color_b[train_t].mean(0)
        means_sched.step()
        if mcmc:
            strategy.step_post_backward(params, optimizers, state, step, info,
                                        lr=optimizers["means"].param_groups[0]["lr"])
        else:
            strategy.step_post_backward(params, optimizers, state, step, info, packed=False)
        if step == prune_step:
            with torch.no_grad():
                seen = torch.zeros(len(params["means"]), dtype=torch.int32, device=device)
                for j in train_ids:
                    o, _, inf = render(j, min(step // 1000, deg), "RGB+ED", train_pose=True)
                    seen += visible_in_view(params["means"], view_of(j, True), Ks_t[j], inf["radii"][0], o[..., 3]).int()
                drop = seen < args.min_views
            from gsplat.strategy.ops import remove

            n_before = len(drop)
            # MCMC's only state ("binoms") is not per-Gaussian; the default strategy's is
            remove(params, optimizers, {} if mcmc else state, drop)
            prune = {"min_views": args.min_views, "step": step, "removed": int(drop.sum()), "kept": n_before - int(drop.sum())}
            print(f"[train] min-views prune at step {step}: removed {prune['removed']} of {n_before} Gaussians", flush=True)
        if len(params["means"]) == 0:
            raise RuntimeError(f"all Gaussians were pruned at step {step}; poses or scene scale are likely wrong")
        if step % args.log_every == 0 or step == args.steps - 1:
            mem = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
            print(f"[train] step {step:6d}  loss {loss.item():.4f}  gaussians {len(params['means']):8d}  "
                  f"{time.time() - t_train:7.1f}s  peak mem {mem:.1f} GB", flush=True)
    train_seconds = time.time() - t_train

    # Evaluate on held-out views (or a couple of train views) and save side-by-side previews.
    psnrs, psnrs_obj, psnrs_cm = [], [], []
    preview_ids = test_ids or [0, n // 2]
    with torch.no_grad():
        for k, i in enumerate(preview_ids):
            img, _, _ = render(i, deg)
            img = img.clamp(0, 1)
            gt = images[i].float() / 255.0
            if test_ids:
                psnrs.append(float(-10 * torch.log10(F.mse_loss(img, gt))))
                psnrs_cm.append(color_matched_psnr(img, gt))
                if masks[i] is not None:
                    psnrs_obj.append(float(-10 * torch.log10(masked_mean((img - gt) ** 2, masks[i]))))
            if k < 4:
                from PIL import Image

                pair = torch.cat([gt, img], 1).mul(255).byte().cpu().numpy()
                Image.fromarray(pair).save(out / "previews" / f"view_{i:04d}_gt_vs_render.png")

    # Rooms: how much of the view from each capture spot sees through the splat (the quality KPI).
    holes = None
    from .scene import capture_mode, floor_height

    if capture_mode(manifest) == "scene":
        from .coverage import hole_report
        from .validate import load_points

        def render_np(vm, K, w, h):
            with torch.no_grad():
                img, alpha, _ = raster(torch.tensor(vm, dtype=torch.float32, device=device),
                                       torch.tensor(K, dtype=torch.float32, device=device), w, h, deg)
            return img.clamp(0, 1).cpu().numpy(), alpha.cpu().numpy()

        floor_y = floor_height(manifest, load_points(args.manifest, manifest), centers)
        holes = hole_report(render_np, manifest, floor_y, out / "previews", camera_centers=centers)
        print(f"[train] holes seen from the capture spots: {holes}", flush=True)

    ply_path = out / "splat.ply"
    with torch.no_grad():
        quats = F.normalize(params["quats"], dim=-1)
        write_gaussians_ply(ply_path, params["means"].cpu().numpy(), params["scales"].cpu().numpy(),
                            quats.cpu().numpy(), params["opacities"].cpu().numpy(),
                            params["sh0"][:, 0].cpu().numpy(), params["shN"].cpu().numpy())

    from .postprocess import postprocess

    object_stats = postprocess(ply_path, args.manifest, out / "splat_object.ply")

    pose_shift = None
    if pose_delta is not None and train_ids:
        d = pose_delta.detach()[train_ids]
        pose_shift = {"rot_deg_max": round(float(d[:, :3].norm(dim=-1).max()) * 180 / np.pi, 3),
                      "trans_cm_max": round(float(d[:, 3:].norm(dim=-1).max()) * 100, 2)}
    metrics = {
        "capture_mode": profile,
        "psnr_test_mean": float(np.mean(psnrs)) if psnrs else None,
        "psnr_test": psnrs,
        "psnr_test_color_matched_mean": float(np.mean(psnrs_cm)) if psnrs_cm else None,
        "psnr_object_mean": float(np.mean(psnrs_obj)) if psnrs_obj else None,
        "psnr_object": psnrs_obj,
        "focus_region": None if region is None else {"center": [round(float(c), 4) for c in region[0]],
                                                     "radius": round(region[1], 4)},
        "num_gaussians": len(params["means"]),
        "steps": args.steps,
        "strategy": args.strategy,
        "depth_priors": n_priors,
        "appearance": args.appearance,
        "pose_opt": pose_shift,
        "rasterize_mode": raster_mode,
        "min_views_prune": prune,
        "holes": holes,
        "train_seconds": round(train_seconds, 1),
        "total_seconds": round(time.time() - t0, 1),
        "frames_train": len(train_ids),
        "frames_test": len(test_ids),
        "image_size": [int(images[0].shape[1]), int(images[0].shape[0])],
        "init": init_src,
        "splat_ply_mb": round(ply_path.stat().st_size / 1e6, 1),
        "splat_object": object_stats,
        "gsplat": gsplat.__version__,
        "torch": torch.__version__,
        "device": torch.cuda.get_device_name() if device == "cuda" else "cpu (dry run)",
        "dry_run": bool(args.dry_run),
    }
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print("[train] done:", json.dumps(metrics, indent=2), flush=True)
    return metrics


def holes_for_splat(splat_ply, manifest_path, out_dir) -> dict:
    """Hole report for an existing splat (e.g. an older run, to compare): needs CUDA. Use the trainer's
    splat.ply (training coordinates), not splat_object.ply (shifted so the floor is at y = 0)."""
    import gsplat

    from .coverage import hole_report
    from .ply_io import read_gaussians_ply
    from .scene import floor_height
    from .validate import load_points

    frames, manifest = load_frames(manifest_path)
    centers = np.stack([f.c2w_gl[:3, 3] for f in frames])
    g = read_gaussians_ply(splat_ply)
    dev = "cuda"
    t = {k: torch.tensor(v, dtype=torch.float32, device=dev) for k, v in g.items()}
    colors = torch.cat([t["sh0"][:, None, :], t["shN"]], 1)
    deg = int(round(np.sqrt(colors.shape[1]))) - 1

    def render_np(vm, K, w, h):
        with torch.no_grad():
            img, alpha, _ = gsplat.rasterization(
                means=t["means"], quats=F.normalize(t["quats_wxyz"], dim=-1), scales=torch.exp(t["scales_log"]),
                opacities=torch.sigmoid(t["opacities_logit"]), colors=colors,
                viewmats=torch.tensor(vm, dtype=torch.float32, device=dev)[None],
                Ks=torch.tensor(K, dtype=torch.float32, device=dev)[None], width=w, height=h, sh_degree=deg)
        return img[0].clamp(0, 1).cpu().numpy(), alpha[0, ..., 0].cpu().numpy()

    Path(out_dir).mkdir(parents=True, exist_ok=True)
    floor_y = floor_height(manifest, load_points(manifest_path, manifest), centers)
    return hole_report(render_np, manifest, floor_y, out_dir, camera_centers=centers)

In [ ]:
%%writefile /kaggle/working/gs_pipeline/train_config.py
"""Training CLI arguments (kept free of torch imports so the CLI loads on CPU)."""

# Defaults per capture type. Objects: 15k steps reached ~90% of 30k quality in a third of the time.
# Scenes (rooms) have far more surface to fit, so they get the full 30k schedule, slightly smaller
# images (more frames) and a Gaussian cap that keeps a T4 (16 GB) from running out of memory.
# The first room (plain 3DGS) had holes on the blank ceiling and needle "fur" wherever few photos looked,
# so scenes also get: MCMC densification with scale/opacity regularisation, monocular depth priors
# (dense init + depth loss, from `gs_pipeline depth`), per-photo exposure compensation, pose refinement,
# anti-aliased rendering and a prune of Gaussians seen by fewer than 3 photos.
PROFILES = {
    "object": {"steps": 15000, "max_side": 1920, "cap_max": 0, "max_init_points": 300_000, "strategy": "default",
               "depth": "off", "appearance": "off", "pose_opt": "off", "antialiased": "off", "min_views": 0},
    "scene": {"steps": 30000, "max_side": 1600, "cap_max": 3_000_000, "max_init_points": 1_000_000,
              "strategy": "mcmc", "depth": "auto", "appearance": "on", "pose_opt": "on", "antialiased": "on",
              "min_views": 3},
}


def resolve_profile(args, manifest: dict) -> str:
    """Fill unset steps / max_side / cap_max from the profile (auto = from the manifest's capture_mode)."""
    name = args.profile
    if name == "auto":
        name = "scene" if manifest.get("capture_mode") == "scene" else "object"
    for key, value in PROFILES[name].items():
        if getattr(args, key) is None:
            setattr(args, key, value)
    return name


def add_train_args(p):
    p.add_argument("manifest")
    p.add_argument("--out", default="out", help="output folder (splat.ply, metrics.json, previews/)")
    p.add_argument("--profile", choices=["auto", "object", "scene"], default="auto",
                   help="defaults for steps/max-side/cap-max; auto picks from the manifest's capture_mode")
    p.add_argument("--steps", type=int, default=None, help="default: from the profile (object 15k, scene 30k)")
    p.add_argument("--max-side", type=int, default=None,
                   help="downscale images so the long side <= this (default: object 1920, scene 1600)")
    p.add_argument("--cap-max", type=int, default=None, help="stop densifying above this many Gaussians (0 = no cap)")
    p.add_argument("--sh-degree", type=int, default=3)
    p.add_argument("--test-every", type=int, default=8, help="hold out every Nth frame for PSNR (0 = none)")
    p.add_argument("--init-random", type=int, default=100_000, help="random points if no points3d.ply")
    p.add_argument("--max-init-points", type=int, default=None, help="default: object 300k, scene 1M")
    p.add_argument("--strategy", choices=["default", "mcmc"], default=None,
                   help="densification: default = 3DGS clone/split, mcmc = 3DGS-MCMC (default: object default, scene mcmc)")
    p.add_argument("--opacity-reg", type=float, default=0.01, help="MCMC only: penalty on mean opacity")
    p.add_argument("--scale-reg", type=float, default=0.01, help="MCMC only: penalty on mean scale (stops needles)")
    p.add_argument("--depth", choices=["auto", "off"], default=None,
                   help="auto: use depth priors from --depth-dir if `gs_pipeline depth` was run (default: scene auto)")
    p.add_argument("--depth-dir", default=None, help="output of `gs_pipeline depth` (default: <bundle>/depth)")
    p.add_argument("--depth-lambda", type=float, default=0.2, help="depth loss weight at the start")
    p.add_argument("--depth-lambda-final", type=float, default=0.01, help="depth loss weight at the end")
    p.add_argument("--appearance", choices=["on", "off"], default=None,
                   help="learn a per-photo colour transform (phone auto-exposure / white balance)")
    p.add_argument("--pose-opt", choices=["on", "off"], default=None, help="refine training camera poses")
    p.add_argument("--antialiased", choices=["on", "off"], default=None, help="anti-aliased rasterisation")
    p.add_argument("--min-views", type=int, default=None,
                   help="after training drop Gaussians seen (unoccluded) by fewer training photos (0 = off)")
    p.add_argument("--ssim-lambda", type=float, default=0.2)
    p.add_argument("--log-every", type=int, default=500)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--focus", choices=["auto", "off"], default="auto",
                   help="auto: only densify inside a sphere around the object (needs object_bounds); off: everywhere")
    p.add_argument("--focus-scale", type=float, default=2.0, help="focus sphere = this x object bounding sphere")
    p.add_argument("--dry-run", action="store_true", help="CPU smoke test with a fake rasterizer (no CUDA)")
    return p

In [ ]:
%%writefile /kaggle/working/gs_pipeline/validate.py
"""CPU-only checks that a capture bundle is trainable.

Errors = the bundle must not be sent to the GPU. Warnings = it will train, but
quality may suffer. The look-at check catches flipped axis conventions: if the
object ends up behind most cameras, the pose math is wrong.
"""

from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

from .cameras import project, viewmat_from_c2w_gl
from .manifest import INTRINSIC_KEYS, frame_intrinsics, load_frames, points_path, read_manifest, schema_errors

MIN_FRAMES = 20


@dataclass
class ValidationReport:
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)
    stats: dict = field(default_factory=dict)

    @property
    def ok(self) -> bool:
        return not self.errors

    def summary(self) -> str:
        lines = [f"{'OK' if self.ok else 'FAILED'}: {len(self.errors)} error(s), {len(self.warnings)} warning(s)"]
        lines += [f"  stat  {k}: {v}" for k, v in self.stats.items()]
        lines += [f"  ERROR {e}" for e in self.errors]
        lines += [f"  WARN  {w}" for w in self.warnings]
        return "\n".join(lines)


def load_points(manifest_path, manifest: dict) -> np.ndarray | None:
    pts = points_path(manifest_path, manifest)
    if pts is None or not pts.exists():
        return None
    from .ply_io import read_points_ply

    return read_points_ply(pts)[0]


def object_bounds(manifest_path, manifest: dict):
    """Object cylinder refined from AR points (see bounds.py), or None if the capture has no bounds."""
    from .bounds import refine_bounds

    return refine_bounds(manifest, load_points(manifest_path, manifest))


def scene_target(manifest: dict, manifest_path, c2ws: list[np.ndarray]) -> tuple[np.ndarray, str]:
    """Point the cameras should be looking at: object middle, else median of points, else ray intersection."""
    b = object_bounds(manifest_path, manifest)
    if b is not None:
        return b.middle, f"object_bounds ({b.source})"
    xyz = load_points(manifest_path, manifest)
    if xyz is not None and len(xyz):
        return np.median(xyz, axis=0), "points3d median"
    # Least-squares point closest to all optical axes.
    A, b = np.zeros((3, 3)), np.zeros(3)
    for c2w in c2ws:
        d = -c2w[:3, 2] / np.linalg.norm(c2w[:3, 2])
        P = np.eye(3) - np.outer(d, d)
        A += P
        b += P @ c2w[:3, 3]
    return np.linalg.lstsq(A, b, rcond=None)[0], "optical-axes intersection"


def validate_manifest(manifest_path, check_files: bool = True) -> ValidationReport:
    manifest_path = Path(manifest_path)
    rep = ValidationReport()
    try:
        manifest = read_manifest(manifest_path)
    except Exception as exc:  # noqa: BLE001 - any parse failure is a validation error
        rep.errors.append(f"cannot read manifest: {exc}")
        return rep

    rep.errors += [f"schema: {e}" for e in schema_errors(manifest)]
    if rep.errors:
        return rep

    frames_raw = manifest["frames"]
    for i, f in enumerate(frames_raw):
        missing = [k for k in INTRINSIC_KEYS if k not in frame_intrinsics(manifest, f)]
        if missing:
            rep.errors.append(f"frame {i}: missing intrinsics {missing}")
    if rep.errors:
        return rep

    frames, _ = load_frames(manifest_path)
    rep.stats["frames"] = len(frames)
    if len(frames) < MIN_FRAMES:
        rep.warnings.append(f"only {len(frames)} frames; >= {MIN_FRAMES} recommended")

    paths = [f["file_path"] for f in frames_raw]
    dupes = {p for p in paths if paths.count(p) > 1}
    if dupes:
        rep.errors.append(f"duplicate file_path entries: {sorted(dupes)[:5]}")

    if any(abs(manifest.get(k, 0.0)) > 1e-6 for k in ("k1", "k2", "p1", "p2")):
        rep.warnings.append("lens distortion coefficients present; trainer treats images as undistorted pinhole")

    sizes = set()
    for fr in frames:
        tag = f"frame {fr.index} ({fr.image_path.name})"
        M = fr.c2w_gl
        if not np.all(np.isfinite(M)):
            rep.errors.append(f"{tag}: non-finite values in transform_matrix")
            continue
        if not np.allclose(M[3], [0, 0, 0, 1], atol=1e-6):
            rep.errors.append(f"{tag}: last row of transform_matrix must be [0,0,0,1]")
        R = M[:3, :3]
        if np.abs(R.T @ R - np.eye(3)).max() > 1e-3 or abs(np.linalg.det(R) - 1) > 1e-3:
            rep.errors.append(f"{tag}: rotation is not orthonormal with det +1 (scaled or mirrored pose)")
        fx, fy, cx, cy = fr.K[0, 0], fr.K[1, 1], fr.K[0, 2], fr.K[1, 2]
        if not (0 < cx < fr.width and 0 < cy < fr.height):
            rep.errors.append(f"{tag}: principal point ({cx:.1f},{cy:.1f}) outside {fr.width}x{fr.height}")
        if not (0.25 < fx / fr.width < 5 and 0.9 < fx / fy < 1.1):
            rep.warnings.append(f"{tag}: unusual focal length fx={fx:.1f} fy={fy:.1f} for width {fr.width}")
        sizes.add((fr.width, fr.height))
        if check_files:
            if not fr.image_path.exists():
                rep.errors.append(f"{tag}: image file missing: {fr.image_path}")
            elif len(sizes) <= 3:  # spot-check the declared size against the real file
                from PIL import Image

                with Image.open(fr.image_path) as im:
                    if im.size != (fr.width, fr.height):
                        rep.errors.append(f"{tag}: image is {im.size[0]}x{im.size[1]} but manifest says "
                                          f"{fr.width}x{fr.height} (rescale intrinsics)")
    rep.stats["resolutions"] = sorted(sizes)
    if rep.errors:
        return rep

    c2ws = [fr.c2w_gl for fr in frames]
    from .scene import capture_mode

    rep.stats["capture mode"] = capture_mode(manifest)
    if capture_mode(manifest) == "scene":
        _validate_scene(rep, manifest, c2ws)
        return rep
    target, how = scene_target(manifest, manifest_path, c2ws)
    centers = np.array([c[:3, 3] for c in c2ws])
    dists = np.linalg.norm(centers - target, axis=1)
    rep.stats["look-at target"] = f"{np.round(target, 3).tolist()} ({how})"
    rep.stats["camera distance to target (min/median/max)"] = (
        f"{dists.min():.3f} / {np.median(dists):.3f} / {dists.max():.3f}")

    in_view = 0
    for fr in frames:
        uv, depth = project(target[None], viewmat_from_c2w_gl(fr.c2w_gl), fr.K)
        if depth[0] > 0 and 0 <= uv[0, 0] < fr.width and 0 <= uv[0, 1] < fr.height:
            in_view += 1
    frac = in_view / len(frames)
    rep.stats["frames seeing target"] = f"{in_view}/{len(frames)}"
    if frac < 0.5:
        rep.errors.append(f"target is visible in only {frac:.0%} of frames; poses are probably in the wrong "
                          "convention (expected OpenGL camera-to-world: camera looks down -z, y up)")
    elif frac < 0.9:
        rep.warnings.append(f"target visible in only {frac:.0%} of frames")

    # Angular coverage around the target (azimuth in the plane orthogonal to the mean up vector).
    up = np.mean([c[:3, 1] for c in c2ws], axis=0)
    up /= np.linalg.norm(up)
    rel = centers - target
    rel_flat = rel - np.outer(rel @ up, up)
    ref = rel_flat[np.argmax(np.linalg.norm(rel_flat, axis=1))]
    ref /= np.linalg.norm(ref)
    az = np.degrees(np.arctan2(rel_flat @ np.cross(up, ref), rel_flat @ ref)) % 360
    covered = len(np.unique((az // 30).astype(int) % 12))
    rep.stats["azimuth coverage (30deg bins)"] = f"{covered}/12"
    if covered < 9:
        rep.warnings.append(f"azimuth coverage only {covered}/12 bins of 30deg; orbit the object fully")
    return rep


def _validate_scene(rep: ValidationReport, manifest: dict, c2ws: list[np.ndarray]) -> None:
    """Room captures: no single target, but the lens must have moved (parallax) and looked all around."""
    from .scene import forward_gl, parallax_check, scene_info

    centers = np.array([c[:3, 3] for c in c2ws])
    errors, warnings, spread = parallax_check(centers)
    rep.errors += errors
    rep.warnings += warnings
    rep.stats["camera spread (m)"] = round(spread, 3)
    stations = scene_info(manifest).get("stations") or []
    rep.stats["stations"] = len(stations)
    if stations and len(stations) < 3:
        rep.warnings.append(f"only {len(stations)} station(s); 4-6 spots around the room give the best result")
    fwd = np.array([forward_gl(c) for c in c2ws])
    az = np.degrees(np.arctan2(fwd[:, 0], -fwd[:, 2])) % 360
    covered = len(np.unique((az // 30).astype(int) % 12))
    rep.stats["view directions (30deg bins)"] = f"{covered}/12"
    if covered < 9:
        rep.warnings.append(f"photos look in only {covered}/12 directions; turn all the way around")

In [ ]:
%%writefile /kaggle/working/gs_pipeline/visualize.py
"""Plot camera frustums + sparse points to a PNG, to eyeball pose sanity on CPU."""

from pathlib import Path

import numpy as np

from .manifest import load_frames, points_path
from .validate import scene_target


def _frustum_lines(c2w_gl, K, w, h, size):
    """World-space line segments of a camera frustum (OpenGL: looks down -z)."""
    corners_px = np.array([[0, 0], [w, 0], [w, h], [0, h]], float)
    dirs = np.stack([(corners_px[:, 0] - K[0, 2]) / K[0, 0],
                     -(corners_px[:, 1] - K[1, 2]) / K[1, 1],  # image y down -> camera y up
                     -np.ones(4)], axis=1) * size
    world = dirs @ c2w_gl[:3, :3].T + c2w_gl[:3, 3]
    o = c2w_gl[:3, 3]
    segs = [(o, c) for c in world] + [(world[i], world[(i + 1) % 4]) for i in range(4)]
    up_tip = (world[0] + world[1]) / 2 + c2w_gl[:3, 1] * size * 0.3  # marks the image top
    segs.append(((world[0] + world[1]) / 2, up_tip))
    return segs


def plot_cameras(manifest_path, out_png, max_points: int = 20000) -> Path:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    frames, manifest = load_frames(manifest_path)
    c2ws = [f.c2w_gl for f in frames]
    target, _ = scene_target(manifest, manifest_path, c2ws)
    centers = np.array([c[:3, 3] for c in c2ws])
    size = 0.12 * np.median(np.linalg.norm(centers - target, axis=1))

    pts = None
    pp = points_path(manifest_path, manifest)
    if pp is not None and pp.exists():
        from .ply_io import read_points_ply

        xyz, rgb = read_points_ply(pp)
        keep = np.linalg.norm(xyz - target, axis=1) < 2.5 * np.median(np.linalg.norm(centers - target, axis=1))
        xyz, rgb = xyz[keep], (rgb[keep] if rgb is not None else None)
        idx = np.random.default_rng(0).choice(len(xyz), min(max_points, len(xyz)), replace=False)
        pts = (xyz[idx], rgb[idx] / 255.0 if rgb is not None else "gray")

    fig = plt.figure(figsize=(14, 7))
    for k, (elev, azim, title) in enumerate([(25, -60, "perspective"), (89, -90, "top-down")]):
        ax = fig.add_subplot(1, 2, k + 1, projection="3d")
        if pts is not None:
            ax.scatter(*pts[0].T, c=pts[1], s=0.3, alpha=0.6)
        for f in frames:
            for a, b in _frustum_lines(f.c2w_gl, f.K, f.width, f.height, size):
                ax.plot(*np.stack([a, b]).T, color="tab:red", lw=0.6)
        ax.scatter(*target, color="tab:blue", s=60, marker="*")
        span = np.ptp(np.vstack([centers, target[None]]), axis=0).max() / 2 * 1.1
        mid = (centers.max(0) + centers.min(0)) / 2
        ax.set_xlim(mid[0] - span, mid[0] + span)
        ax.set_ylim(mid[1] - span, mid[1] + span)
        ax.set_zlim(mid[2] - span, mid[2] + span)
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(f"{title}: {len(frames)} cameras (red), target (blue star)")
        ax.set_xlabel("x"), ax.set_ylabel("y"), ax.set_zlabel("z")
    fig.tight_layout()
    out_png = Path(out_png)
    fig.savefig(out_png, dpi=110)
    plt.close(fig)
    return out_png

## 4. Find and unpack the capture

In [ ]:
import glob, os, zipfile
zips = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True))
manifests = sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))
if zips:
    print("using", zips[0])
    zipfile.ZipFile(zips[0]).extractall("/tmp/capture")
    CAPTURE = "/tmp/capture"
elif manifests:  # Kaggle usually auto-extracts uploaded zips
    CAPTURE = os.path.dirname(manifests[0])
else:
    raise FileNotFoundError("No capture found: add your capture dataset via 'Add Input'")
print("capture folder:", CAPTURE, "|", len(glob.glob(CAPTURE + "/images/*.jpg")), "images")

## 4b. Pose refinement (skipped if the bundle was already refined on the PC)

In [ ]:
import json, subprocess, sys
source = json.load(open(CAPTURE + "/manifest.json")).get("source", "")
if "colmap" in source:
    print("already refined:", source)
else:
    !pip install -q pycolmap opencv-python-headless
    !python -m gs_pipeline refine {CAPTURE}/manifest.json --out /tmp/refined --threads 4
    report = json.load(open("/tmp/refined/refine_report.json"))
    CAPTURE = str(__import__("pathlib").Path(report["manifest"]).parent)
    print("training on:", CAPTURE, "| fallback to AR poses:", report["fallback"])

## 5. CPU checks: validate (must say capture mode: scene) + camera plot

In [ ]:
%cd /kaggle/working
!python -m gs_pipeline validate {CAPTURE}/manifest.json
!python -m gs_pipeline plot {CAPTURE}/manifest.json --out /kaggle/working/cameras.png
from IPython.display import Image, display
display(Image("/kaggle/working/cameras.png"))

## 5b. Depth priors (Depth-Anything-V2, ~3-5 min)
Blank ceilings and walls give COLMAP almost no points, so plain 3DGS leaves holes or "fur" there.
Each photo's predicted depth is aligned to the sparse points; the result seeds Gaussians on those
surfaces and supervises their shape. Check `aligned` (should be close to `frames`) and `median_rel_err`
(below ~0.1 is good).

In [ ]:
import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.45"):
    !pip install -q "transformers>=4.45"
%cd /kaggle/working
!python -m gs_pipeline depth {CAPTURE}/manifest.json --out /kaggle/working/depth

## 6. Train (scene profile: 30k steps, 1600 px, MCMC with cap 3M, depth loss, per-photo exposure,
pose refinement, anti-aliased; Gaussians seen by < 3 photos are pruned at the end)

In [ ]:
%cd /kaggle/working
!python -m gs_pipeline train {CAPTURE}/manifest.json --out /kaggle/working/out --profile scene --depth-dir /kaggle/working/depth

## 7. Results

In [ ]:
import json, glob, os
from IPython.display import Image, display
print(json.dumps(json.load(open("/kaggle/working/out/metrics.json")), indent=2))
for p in sorted(glob.glob("/kaggle/working/out/previews/*.png")):
    print(os.path.basename(p)); display(Image(p))
for p in sorted(glob.glob("/kaggle/working/out/*")):
    print(f"{os.path.getsize(p) / 1e6:8.1f} MB  {p}") if os.path.isfile(p) else None
print("\nDownload out/splat_object.ply (object only) or out/splat.ply (whole scene) from the Output panel")
print("and drop it into https://superspl.at/editor")

## 8. (Optional) Compare with an earlier run
Add the earlier run's notebook output as an input (*Add Input* → *Your Work* → that notebook version). This
scores its `splat.ply` with the same hole report, so `hole_fraction` can be compared like for like.

In [ ]:
import glob, json
old = [p for p in glob.glob("/kaggle/input/**/splat.ply", recursive=True)]
if not old:
    print("no earlier splat.ply attached - skipping")
else:
    print("earlier splat:", old[0])
    !python -m gs_pipeline holes {old[0]} {CAPTURE}/manifest.json --out /kaggle/working/holes_old
    new = json.load(open("/kaggle/working/out/metrics.json"))
    print("this run's hole_fraction:", new["holes"]["hole_fraction"], "| per spot:", new["holes"]["hole_fraction_per_station"])